Cell 0  → verification / R0 dependency gate
Cell 1  → fold-aware TF-IDF build
Cell 2  → math token extraction
Cell 3  → character similarity
Cell 4  → session-local sparse retrieval
Cell 5  → score fusion
Cell 6  → Top-K selection
Cell 7  → sparse_candidates.parquet
Cell 8  → retrieval diagnostics
Cell 9  → audit + manifest
Cell 10 → R1 FREEZE GATE

In [1]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 0 — ENVIRONMENT + R0 DEPENDENCY VERIFICATION
# ==============================================================================

from pathlib import Path
import hashlib
import json
import platform
import sys
import gc

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 0 — ENVIRONMENT + R0 DEPENDENCY VERIFICATION")
print("=" * 80)


# ==============================================================================
# 1. ENVIRONMENT
# ==============================================================================

print("\n" + "=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("Python :", sys.version)
print("OS     :", platform.platform())
print("pandas :", pd.__version__)
print("pyarrow:", pa.__version__)
print("CWD    :", Path.cwd())


# ==============================================================================
# 2. PROJECT ROOT DISCOVERY
# ==============================================================================

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = None

for candidate in [
    NOTEBOOK_DIR,
    *NOTEBOOK_DIR.parents,
]:
    if (
        (candidate / "Dataset").exists()
        and (candidate / "scratch_mastery_outputs").exists()
    ):
        PROJECT_ROOT = candidate
        break


assert PROJECT_ROOT is not None, (
    "Could not discover Trace-the-Ace project root."
)


DATASET_DIR = PROJECT_ROOT / "Dataset"
SCRATCH_ROOT = PROJECT_ROOT / "scratch_mastery_outputs"


print("\n" + "=" * 80)
print("PROJECT PATHS")
print("=" * 80)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATASET_DIR  :", DATASET_DIR)
print("SCRATCH_ROOT :", SCRATCH_ROOT)


assert PROJECT_ROOT.exists()
assert DATASET_DIR.exists()
assert SCRATCH_ROOT.exists()


# ==============================================================================
# 3. R0 PATHS
# ==============================================================================

R0_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R0_input"
)

R0_RETRIEVAL_QUERIES = (
    R0_ROOT
    / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT
    / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE = (
    R0_ROOT
    / "objective_catalogue.parquet"
)

R0_MANIFEST = (
    R0_ROOT
    / "r0_manifest.json"
)


print("\n" + "=" * 80)
print("R0 DEPENDENCIES")
print("=" * 80)

r0_paths = {
    "R0_ROOT": R0_ROOT,
    "retrieval_queries": R0_RETRIEVAL_QUERIES,
    "session_turn_index": R0_SESSION_TURN_INDEX,
    "objective_catalogue": R0_OBJECTIVE_CATALOGUE,
    "r0_manifest": R0_MANIFEST,
}

for name, path in r0_paths.items():
    print(
        f"{name:24s}: "
        f"{path} | exists={path.exists()}"
    )

assert R0_ROOT.exists(), (
    "R0 input directory does not exist."
)

assert R0_RETRIEVAL_QUERIES.exists(), (
    "R0 retrieval_queries.parquet is missing."
)

assert R0_SESSION_TURN_INDEX.exists(), (
    "R0 session_turn_index.parquet is missing."
)

assert R0_OBJECTIVE_CATALOGUE.exists(), (
    "R0 objective_catalogue.parquet is missing."
)

assert R0_MANIFEST.exists(), (
    "R0 manifest is missing."
)


# ==============================================================================
# 4. LOAD R0 MANIFEST
# ==============================================================================

with open(
    R0_MANIFEST,
    "r",
    encoding="utf-8",
) as f:
    R0_MANIFEST_DATA = json.load(f)


print("\n" + "=" * 80)
print("R0 MANIFEST")
print("=" * 80)

print(
    "stage  :",
    R0_MANIFEST_DATA.get("stage"),
)

print(
    "status :",
    R0_MANIFEST_DATA.get("status"),
)

print(
    "r0_ready :",
    R0_MANIFEST_DATA.get("r0_ready"),
)


assert (
    R0_MANIFEST_DATA.get("stage")
    == "R0_RETRIEVAL_INPUT_BUILDER"
), (
    "R0 manifest stage is not correct."
)

assert (
    R0_MANIFEST_DATA.get("status")
    == "READY"
), (
    "R0 manifest does not report READY."
)

assert (
    R0_MANIFEST_DATA.get("r0_ready")
    is True
), (
    "R0 manifest does not report r0_ready=True."
)


# ==============================================================================
# 5. SHA256 HELPER
# ==============================================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ==============================================================================
# 6. VERIFY R0 ARTIFACT HASHES
# ==============================================================================

print("\n" + "=" * 80)
print("R0 ARTIFACT SHA256 VERIFICATION")
print("=" * 80)


r0_artifacts = {
    "retrieval_queries": R0_RETRIEVAL_QUERIES,
    "session_turn_index": R0_SESSION_TURN_INDEX,
    "objective_catalogue": R0_OBJECTIVE_CATALOGUE,
}

for name, path in r0_artifacts.items():

    manifest_artifact = (
        R0_MANIFEST_DATA
        .get("artifacts", {})
        .get(name)
    )

    assert manifest_artifact is not None, (
        f"Manifest entry missing for {name}."
    )

    expected_sha = manifest_artifact.get(
        "sha256"
    )

    assert expected_sha, (
        f"Expected SHA256 missing for {name}."
    )

    observed_sha = sha256_file(path)

    print(f"\n{name}")
    print("Expected:", expected_sha)
    print("Observed:", observed_sha)
    print(
        "Match   :",
        observed_sha == expected_sha,
    )

    assert observed_sha == expected_sha, (
        f"R0 artifact SHA256 mismatch: {name}"
    )


# ==============================================================================
# 7. LOAD R0 ARTIFACTS
#
# R1 works from the frozen R0 artifacts.
# Canonical Phase-1 data is NOT modified.
# ==============================================================================

retrieval_queries = pd.read_parquet(
    R0_RETRIEVAL_QUERIES
)

session_turn_index = pd.read_parquet(
    R0_SESSION_TURN_INDEX
)

objective_catalogue = pd.read_parquet(
    R0_OBJECTIVE_CATALOGUE
)


# ==============================================================================
# 8. R0 POPULATION CONTRACT
# ==============================================================================

print("\n" + "=" * 80)
print("R0 POPULATION CONTRACT")
print("=" * 80)

print(
    "Retrieval queries :",
    f"{len(retrieval_queries):,}",
)

print(
    "Session-turn index:",
    f"{len(session_turn_index):,}",
)

print(
    "Objective catalogue:",
    f"{len(objective_catalogue):,}",
)


assert len(retrieval_queries) == 35072, (
    "R0 retrieval query population changed."
)

assert len(session_turn_index) == 6139854, (
    "R0 session-turn population changed."
)

assert len(objective_catalogue) == 398, (
    "R0 objective population changed."
)


# ==============================================================================
# 9. REQUIRED R1 INPUT SCHEMA
# ==============================================================================

print("\n" + "=" * 80)
print("R1 INPUT SCHEMA")
print("=" * 80)


REQUIRED_QUERY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_raw",
    "objective_uid",
    "fold",
]

REQUIRED_TURN_COLUMNS = [
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
]

REQUIRED_OBJECTIVE_COLUMNS = [
    "objective_uid",
    "objective_raw",
]


missing_query_columns = [
    c
    for c in REQUIRED_QUERY_COLUMNS
    if c not in retrieval_queries.columns
]

missing_turn_columns = [
    c
    for c in REQUIRED_TURN_COLUMNS
    if c not in session_turn_index.columns
]

missing_objective_columns = [
    c
    for c in REQUIRED_OBJECTIVE_COLUMNS
    if c not in objective_catalogue.columns
]


print(
    "Missing query columns    :",
    missing_query_columns,
)

print(
    "Missing turn columns     :",
    missing_turn_columns,
)

print(
    "Missing objective columns:",
    missing_objective_columns,
)


assert missing_query_columns == [], (
    f"R0 query schema missing: {missing_query_columns}"
)

assert missing_turn_columns == [], (
    f"R0 turn schema missing: {missing_turn_columns}"
)

assert missing_objective_columns == [], (
    f"R0 objective schema missing: "
    f"{missing_objective_columns}"
)


# ==============================================================================
# 10. TARGET / LABEL ISOLATION
# ==============================================================================

print("\n" + "=" * 80)
print("TARGET / LABEL ISOLATION")
print("=" * 80)


PROHIBITED_R1_INPUT_FIELDS = {
    "target",
    "label",
    "prediction",
    "positive_rate",
    "target_mean",
    "target_count",
    "positive_count",
    "negative_count",
}


query_prohibited = sorted(
    set(
        c.lower()
        for c in retrieval_queries.columns
    )
    & PROHIBITED_R1_INPUT_FIELDS
)

turn_prohibited = sorted(
    set(
        c.lower()
        for c in session_turn_index.columns
    )
    & PROHIBITED_R1_INPUT_FIELDS
)

objective_prohibited = sorted(
    set(
        c.lower()
        for c in objective_catalogue.columns
    )
    & PROHIBITED_R1_INPUT_FIELDS
)


print(
    "Query prohibited fields    :",
    query_prohibited,
)

print(
    "Turn prohibited fields     :",
    turn_prohibited,
)

print(
    "Objective prohibited fields:",
    objective_prohibited,
)


assert query_prohibited == [], (
    "Target/evaluation field leaked into R1 query input."
)

assert turn_prohibited == [], (
    "Target/evaluation field leaked into R1 turn input."
)

assert objective_prohibited == [], (
    "Target/evaluation field leaked into R1 objective input."
)


# ==============================================================================
# 11. RESPONSE / OBJECTIVE / SESSION IDENTITY
# ==============================================================================

print("\n" + "=" * 80)
print("IDENTITY CONTRACT")
print("=" * 80)


assert retrieval_queries[
    "response_id"
].is_unique

assert retrieval_queries[
    "session_id"
].notna().all()

assert retrieval_queries[
    "objective_uid"
].notna().all()

assert retrieval_queries[
    "objective_raw"
].notna().all()

assert session_turn_index[
    "turn_uid"
].is_unique

assert session_turn_index[
    "session_id"
].notna().all()

assert objective_catalogue[
    "objective_uid"
].is_unique

assert objective_catalogue[
    "objective_uid"
].notna().all()


print(
    "response_id unique    : True"
)

print(
    "turn_uid unique       : True"
)

print(
    "objective_uid unique  : True"
)

print(
    "Required IDs non-null : True"
)


# ==============================================================================
# 12. RESPONSE → SESSION COVERAGE
# ==============================================================================

query_sessions = set(
    retrieval_queries[
        "session_id"
    ]
)

turn_sessions = set(
    session_turn_index[
        "session_id"
    ]
)

missing_sessions = (
    query_sessions
    - turn_sessions
)

assert len(missing_sessions) == 0, (
    "R1 query contains session without turns."
)


# ==============================================================================
# 13. RESPONSE → OBJECTIVE COVERAGE
# ==============================================================================

query_objectives = set(
    retrieval_queries[
        "objective_uid"
    ]
)

catalogue_objectives = set(
    objective_catalogue[
        "objective_uid"
    ]
)

missing_objectives = (
    query_objectives
    - catalogue_objectives
)

assert len(missing_objectives) == 0, (
    "R1 query references unknown objective."
)


# ==============================================================================
# 14. FOLD CONTRACT
# ==============================================================================

observed_folds = sorted(
    retrieval_queries[
        "fold"
    ].dropna().unique().tolist()
)

expected_folds = [
    0, 1, 2, 3, 4
]


print("\n" + "=" * 80)
print("FOLD CONTRACT")
print("=" * 80)

print(
    "Observed folds:",
    observed_folds,
)

print(
    "Expected folds:",
    expected_folds,
)


assert observed_folds == expected_folds, (
    "R1 retrieval query folds differ from frozen contract."
)


# ==============================================================================
# 15. R1 OUTPUT ROOT
# ==============================================================================

R1_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
    / "R1_sparse"
)

R1_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TFIDF_ROOT = (
    R1_ROOT
    / "tfidf_artifacts"
)

TFIDF_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print("\n" + "=" * 80)
print("R1 OUTPUT ROOT")
print("=" * 80)

print(
    "R1_ROOT:",
    R1_ROOT,
)

print(
    "TFIDF_ROOT:",
    TFIDF_ROOT,
)


# ==============================================================================
# 16. R1 RUNTIME CONTRACT
# ==============================================================================

R1_ENVIRONMENT_READY = True
R1_R0_DEPENDENCY_READY = True
R1_INPUT_SCHEMA_READY = True
R1_TARGET_ISOLATION_READY = True
R1_FOLD_CONTRACT_READY = True


R1_CELL_0_READY = (
    R1_ENVIRONMENT_READY
    and R1_R0_DEPENDENCY_READY
    and R1_INPUT_SCHEMA_READY
    and R1_TARGET_ISOLATION_READY
    and R1_FOLD_CONTRACT_READY
)


# ==============================================================================
# 17. FINAL CELL 0 STATUS
# ==============================================================================

print("\n" + "=" * 80)
print("R1 CELL 0 STATUS")
print("=" * 80)

print(
    "Environment ready          :",
    R1_ENVIRONMENT_READY,
)

print(
    "R0 dependency ready        :",
    R1_R0_DEPENDENCY_READY,
)

print(
    "Input schema ready         :",
    R1_INPUT_SCHEMA_READY,
)

print(
    "Target isolation ready     :",
    R1_TARGET_ISOLATION_READY,
)

print(
    "Fold contract ready        :",
    R1_FOLD_CONTRACT_READY,
)

print(
    "R1_CELL_0_READY            :",
    R1_CELL_0_READY,
)

assert R1_CELL_0_READY is True

print("=" * 80)
print("R1 CELL 0 VERIFICATION: PASS")
print("=" * 80)


# Keep these in memory for Cell 1.
gc.collect()

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 0 — ENVIRONMENT + R0 DEPENDENCY VERIFICATION

ENVIRONMENT
Python : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
OS     : Windows-10-10.0.26200-SP0
pandas : 2.3.3
pyarrow: 25.0.1
CWD    : d:\Competition\Trace-the-race-local\Notebooks

PROJECT PATHS
PROJECT_ROOT : d:\Competition\Trace-the-race-local
DATASET_DIR  : d:\Competition\Trace-the-race-local\Dataset
SCRATCH_ROOT : d:\Competition\Trace-the-race-local\scratch_mastery_outputs

R0 DEPENDENCIES
R0_ROOT                 : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input | exists=True
retrieval_queries       : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\retrieval_queries.parquet | exists=True
session_turn_index      : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet | exists=True
objective_catalogue     : d:\Competiti

0

In [2]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 1 — FOLD-AWARE TF-IDF SPACE BUILD
# ==============================================================================

from pathlib import Path
import json
import gc
import math
import hashlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import HashingVectorizer
from scipy import sparse


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 1 — FOLD-AWARE TF-IDF SPACE BUILD")
print("=" * 80)


# ==============================================================================
# 0. HARD DEPENDENCY GATE
# ==============================================================================

assert R1_CELL_0_READY is True, (
    "R1 Cell 0 verification did not pass."
)

print("\nR1 Cell 0 dependency: PASS")


# ==============================================================================
# 1. CONFIGURATION
#
# Important:
# We use HashingVectorizer instead of materializing a giant vocabulary.
#
# This still gives us:
#     TF representation
#     ×
#     fold-specific IDF
#
# without creating a 6.1M-row dense/single giant count matrix.
# ==============================================================================

R1_TFIDF_ROOT = (
    R1_ROOT
    / "tfidf_artifacts"
)

R1_TFIDF_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# --------------------------------------------------------------------------
# Word-level sparse representation
# --------------------------------------------------------------------------

WORD_N_FEATURES = 2 ** 18

WORD_NGRAM_RANGE = (
    1,
    2,
)

WORD_MIN_DF = 2

WORD_SUBLINEAR_TF = True

# --------------------------------------------------------------------------
# Streaming
# --------------------------------------------------------------------------

TURN_BATCH_SIZE = 50_000

# --------------------------------------------------------------------------
# Numerical dtype
# --------------------------------------------------------------------------

TFIDF_DTYPE = np.float32


print("\n" + "=" * 80)
print("R1 TF-IDF CONFIGURATION")
print("=" * 80)

print(
    "Word hashing dimensions :",
    f"{WORD_N_FEATURES:,}",
)

print(
    "Word n-gram range       :",
    WORD_NGRAM_RANGE,
)

print(
    "Minimum document freq   :",
    WORD_MIN_DF,
)

print(
    "Sublinear TF            :",
    WORD_SUBLINEAR_TF,
)

print(
    "Turn batch size         :",
    f"{TURN_BATCH_SIZE:,}",
)

print(
    "TF-IDF dtype            :",
    TFIDF_DTYPE,
)


# ==============================================================================
# 2. SESSION → FOLD CONTRACT
#
# Sessions are already frozen to one fold in Phase 1.
# We derive the session fold from R0 retrieval queries.
# ==============================================================================

session_fold = (
    retrieval_queries[
        [
            "session_id",
            "fold",
        ]
    ]
    .drop_duplicates()
    .copy()
)

# A session must resolve to exactly one fold.
session_fold_counts = (
    session_fold
    .groupby("session_id")["fold"]
    .nunique()
)

fold_conflicts = (
    session_fold_counts[
        session_fold_counts > 1
    ]
)

assert len(fold_conflicts) == 0, (
    "A session maps to multiple folds."
)

session_fold = (
    session_fold
    .drop_duplicates(
        subset=["session_id"]
    )
)

session_fold_map = dict(
    zip(
        session_fold["session_id"],
        session_fold["fold"],
    )
)


print("\n" + "=" * 80)
print("SESSION → FOLD CONTRACT")
print("=" * 80)

print(
    "Unique sessions :",
    f"{len(session_fold_map):,}",
)

print(
    "Observed folds  :",
    sorted(session_fold["fold"].unique().tolist()),
)

assert (
    len(session_fold_map)
    == retrieval_queries["session_id"].nunique()
), (
    "Session→fold mapping population mismatch."
)

assert sorted(
    session_fold["fold"].unique().tolist()
) == [0, 1, 2, 3, 4]


# ==============================================================================
# 3. CANONICAL TURN PARQUET SOURCE
# ==============================================================================

R0_TURN_SOURCE = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
    / "01_data_foundation"
    / "03_integrity"
    / "canonical"
    / "turns.parquet"
)

assert R0_TURN_SOURCE.exists(), (
    "Canonical turns.parquet not found."
)

turn_pf = pq.ParquetFile(
    R0_TURN_SOURCE
)

turn_total_rows = int(
    turn_pf.metadata.num_rows
)

assert turn_total_rows == 6_139_854, (
    "Canonical turn population changed."
)

print("\nCanonical turn rows:", f"{turn_total_rows:,}")


# ==============================================================================
# 4. HASHING VECTORIZER
#
# alternate_sign=False is mandatory here because the hashed representation
# is going to be used for document-frequency counting and TF-IDF weighting.
# ==============================================================================

word_vectorizer = HashingVectorizer(
    n_features=WORD_N_FEATURES,
    alternate_sign=False,
    norm=None,
    lowercase=True,
    strip_accents="unicode",
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=WORD_NGRAM_RANGE,
    binary=False,
)

print(
    "\nWord HashingVectorizer initialized."
)


# ==============================================================================
# 5. HASHED DOCUMENT-FREQUENCY ACCUMULATION
#
# For each held-out fold:
#
#     IDF corpus = turns from the other 4 folds
#
# This prevents the held-out fold from influencing its retrieval space.
#
# We do NOT use target.
# We do NOT use target-derived statistics.
# ==============================================================================


def accumulate_document_frequency_for_fold(
    heldout_fold: int,
):
    """
    Build document-frequency counts using ONLY turns whose session
    belongs to the four non-held-out folds.

    Returns
    -------
    df_counts : np.ndarray[int64]
        Number of training documents containing each hashed feature.

    n_docs : int
        Number of training documents observed.
    """

    df_counts = np.zeros(
        WORD_N_FEATURES,
        dtype=np.int64,
    )

    n_docs = 0

    for batch_no, batch in enumerate(
        turn_pf.iter_batches(
            batch_size=TURN_BATCH_SIZE,
            columns=[
                "session_id",
                "text_norm",
            ],
        ),
        start=1,
    ):

        batch_df = batch.to_pandas()

        # --------------------------------------------------------------
        # Resolve fold.
        # --------------------------------------------------------------

        batch_folds = (
            batch_df["session_id"]
            .map(session_fold_map)
        )

        assert batch_folds.notna().all(), (
            "Turn contains session without frozen fold."
        )

        keep = (
            batch_folds.to_numpy()
            != heldout_fold
        )

        if not keep.any():
            continue

        texts = (
            batch_df.loc[
                keep,
                "text_norm",
            ]
            .astype(str)
            .tolist()
        )

        if not texts:
            continue

        # --------------------------------------------------------------
        # Hash TF representation.
        # --------------------------------------------------------------

        X = word_vectorizer.transform(
            texts
        )

        # --------------------------------------------------------------
        # Document frequency:
        # each feature counts at most once per document.
        # --------------------------------------------------------------

        X_bin = X.copy()

        if X_bin.nnz:
            X_bin.data = np.ones(
                X_bin.nnz,
                dtype=np.int8,
            )

        batch_df_counts = np.asarray(
            X_bin.sum(axis=0)
        ).ravel()

        df_counts += (
            batch_df_counts
            .astype(np.int64)
        )

        n_docs += X.shape[0]

        del batch_df
        del batch_folds
        del texts
        del X
        del X_bin
        del batch_df_counts

        if batch_no % 25 == 0:
            print(
                f"  fold={heldout_fold} | "
                f"batches={batch_no:,} | "
                f"docs={n_docs:,}"
            )

            gc.collect()

    return df_counts, n_docs


# ==============================================================================
# 6. IDF CONSTRUCTION
# ==============================================================================

def build_idf(
    df_counts,
    n_docs,
):
    """
    Smoothed sklearn-style IDF:

        idf = log((1 + n_docs) / (1 + df)) + 1

    Unseen hashed features receive the maximum IDF.
    """

    assert n_docs > 0

    idf = (
        np.log(
            (1.0 + float(n_docs))
            /
            (1.0 + df_counts.astype(np.float64))
        )
        + 1.0
    )

    return idf.astype(
        TFIDF_DTYPE
    )


# ==============================================================================
# 7. FOLD-BY-FOLD TF-IDF ARTIFACT BUILD
# ==============================================================================

fold_artifact_manifest = {}

print("\n" + "=" * 80)
print("BUILDING FOLD-AWARE TF-IDF SPACES")
print("=" * 80)


for heldout_fold in [0, 1, 2, 3, 4]:

    print("\n" + "-" * 80)
    print(
        f"FOLD {heldout_fold} "
        f"TF-IDF SPACE"
    )
    print("-" * 80)

    fold_dir = (
        R1_TFIDF_ROOT
        / f"fold_{heldout_fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    df_path = (
        fold_dir
        / "word_document_frequency.npy"
    )

    idf_path = (
        fold_dir
        / "word_idf.npy"
    )

    config_path = (
        fold_dir
        / "tfidf_config.json"
    )

    # ------------------------------------------------------------------
    # Compute DF
    # ------------------------------------------------------------------

    df_counts, n_train_docs = (
        accumulate_document_frequency_for_fold(
            heldout_fold
        )
    )

    assert n_train_docs > 0, (
        f"No training documents for held-out fold {heldout_fold}."
    )

    # ------------------------------------------------------------------
    # IDF
    # ------------------------------------------------------------------

    idf = build_idf(
        df_counts,
        n_train_docs,
    )

    assert idf.shape == (
        WORD_N_FEATURES,
    )

    assert np.isfinite(idf).all(), (
        "Non-finite IDF values detected."
    )

    assert (idf >= 1.0).all(), (
        "Unexpected IDF values below 1."
    )

    # ------------------------------------------------------------------
    # Persist arrays
    # ------------------------------------------------------------------

    np.save(
        df_path,
        df_counts,
        allow_pickle=False,
    )

    np.save(
        idf_path,
        idf,
        allow_pickle=False,
    )

    # ------------------------------------------------------------------
    # Configuration
    # ------------------------------------------------------------------

    config = {
        "stage": "R1_SPARSE_RETRIEVAL",
        "representation": "hashed_word_tfidf",
        "vectorizer": "sklearn.HashingVectorizer",
        "n_features": int(
            WORD_N_FEATURES
        ),
        "ngram_range": [
            int(WORD_NGRAM_RANGE[0]),
            int(WORD_NGRAM_RANGE[1]),
        ],
        "min_df_requested": int(
            WORD_MIN_DF
        ),
        "sublinear_tf": bool(
            WORD_SUBLINEAR_TF
        ),
        "lowercase": True,
        "strip_accents": "unicode",
        "alternate_sign": False,
        "norm": None,
        "heldout_fold": int(
            heldout_fold
        ),
        "training_folds": [
            f
            for f in [0, 1, 2, 3, 4]
            if f != heldout_fold
        ],
        "training_turn_rows": int(
            n_train_docs
        ),
        "target_used": False,
        "target_statistics_used": False,
        "session_fold_mapping_source": (
            "R0 retrieval_queries.parquet"
        ),
        "canonical_turn_source": str(
            R0_TURN_SOURCE
        ),
    }

    with open(
        config_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            config,
            f,
            indent=2,
            ensure_ascii=False,
        )

    # ------------------------------------------------------------------
    # File checks
    # ------------------------------------------------------------------

    assert df_path.exists()
    assert idf_path.exists()
    assert config_path.exists()

    fold_artifact_manifest[
        str(heldout_fold)
    ] = {
        "heldout_fold": int(
            heldout_fold
        ),
        "training_folds": [
            int(f)
            for f in [0, 1, 2, 3, 4]
            if f != heldout_fold
        ],
        "training_turn_rows": int(
            n_train_docs
        ),
        "df_path": str(
            df_path
        ),
        "idf_path": str(
            idf_path
        ),
        "config_path": str(
            config_path
        ),
        "df_nonzero_features": int(
            np.count_nonzero(
                df_counts
            )
        ),
        "idf_min": float(
            idf.min()
        ),
        "idf_max": float(
            idf.max()
        ),
    }

    print(
        f"Fold {heldout_fold} PASS"
    )

    print(
        "  training folds      :",
        [f for f in [0, 1, 2, 3, 4] if f != heldout_fold],
    )

    print(
        "  training turn rows  :",
        f"{n_train_docs:,}",
    )

    print(
        "  nonzero hash bins   :",
        f"{np.count_nonzero(df_counts):,}",
    )

    print(
        "  IDF min             :",
        float(idf.min()),
    )

    print(
        "  IDF max             :",
        float(idf.max()),
    )

    # Release large arrays before next fold.
    del df_counts
    del idf

    gc.collect()


# ==============================================================================
# 8. GLOBAL ARTIFACT MANIFEST
# ==============================================================================

R1_TFIDF_MANIFEST = (
    R1_TFIDF_ROOT
    / "tfidf_manifest.json"
)

tfidf_manifest = {
    "stage": "R1_SPARSE_RETRIEVAL",
    "representation": "hashed_word_tfidf",
    "n_features": int(
        WORD_N_FEATURES
    ),
    "ngram_range": [
        int(WORD_NGRAM_RANGE[0]),
        int(WORD_NGRAM_RANGE[1]),
    ],
    "folds": fold_artifact_manifest,
    "target_used": False,
    "target_statistics_used": False,
    "full_corpus_count_matrix_materialized": False,
    "session_local_retrieval_boundary": True,
    "canonical_source_immutable": True,
}


with open(
    R1_TFIDF_MANIFEST,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        tfidf_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert R1_TFIDF_MANIFEST.exists()


# ==============================================================================
# 9. FINAL ARTIFACT VERIFICATION
# ==============================================================================

print("\n" + "=" * 80)
print("R1 CELL 1 FINAL AUDIT")
print("=" * 80)


for heldout_fold in [0, 1, 2, 3, 4]:

    fold_dir = (
        R1_TFIDF_ROOT
        / f"fold_{heldout_fold}"
    )

    df_path = (
        fold_dir
        / "word_document_frequency.npy"
    )

    idf_path = (
        fold_dir
        / "word_idf.npy"
    )

    config_path = (
        fold_dir
        / "tfidf_config.json"
    )

    assert df_path.exists()
    assert idf_path.exists()
    assert config_path.exists()

    df_check = np.load(
        df_path,
        mmap_mode="r",
    )

    idf_check = np.load(
        idf_path,
        mmap_mode="r",
    )

    assert df_check.shape == (
        WORD_N_FEATURES,
    )

    assert idf_check.shape == (
        WORD_N_FEATURES,
    )

    assert np.isfinite(
        idf_check
    ).all()

    assert (
        idf_check >= 1.0
    ).all()

    del df_check
    del idf_check


assert R1_TFIDF_MANIFEST.exists()


# ==============================================================================
# 10. RUNTIME FLAGS
# ==============================================================================

R1_TFIDF_READY = True
R1_FOLD_AWARE_READY = True
R1_TFIDF_TARGET_FREE = True
R1_TFIDF_SERIALIZATION_READY = True

R1_CELL_1_READY = (
    R1_TFIDF_READY
    and R1_FOLD_AWARE_READY
    and R1_TFIDF_TARGET_FREE
    and R1_TFIDF_SERIALIZATION_READY
)


# ==============================================================================
# 11. FINAL STATUS
# ==============================================================================

print("\n" + "=" * 80)
print("R1 CELL 1 STATUS")
print("=" * 80)

print(
    "Fold-aware TF-IDF ready :",
    R1_TFIDF_READY,
)

print(
    "Fold isolation ready    :",
    R1_FOLD_AWARE_READY,
)

print(
    "Target-free             :",
    R1_TFIDF_TARGET_FREE,
)

print(
    "Serialization ready     :",
    R1_TFIDF_SERIALIZATION_READY,
)

print(
    "R1_CELL_1_READY         :",
    R1_CELL_1_READY,
)

assert R1_CELL_1_READY is True

print("=" * 80)
print("R1 CELL 1 — TF-IDF BUILD: PASS")
print("=" * 80)


gc.collect()

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 1 — FOLD-AWARE TF-IDF SPACE BUILD

R1 Cell 0 dependency: PASS

R1 TF-IDF CONFIGURATION
Word hashing dimensions : 262,144
Word n-gram range       : (1, 2)
Minimum document freq   : 2
Sublinear TF            : True
Turn batch size         : 50,000
TF-IDF dtype            : <class 'numpy.float32'>

SESSION → FOLD CONTRACT
Unique sessions : 22,821
Observed folds  : [0, 1, 2, 3, 4]

Canonical turn rows: 6,139,854

Word HashingVectorizer initialized.

BUILDING FOLD-AWARE TF-IDF SPACES

--------------------------------------------------------------------------------
FOLD 0 TF-IDF SPACE
--------------------------------------------------------------------------------
  fold=0 | batches=25 | docs=1,002,693
  fold=0 | batches=50 | docs=2,007,172
  fold=0 | batches=75 | docs=3,004,055
  fold=0 | batches=100 | docs=4,007,821
Fold 0 PASS
  training folds      : [1, 2, 3, 4]
  training turn rows  : 4,917,445
  nonzero hash bins   : 258,843
  IDF min           

143

**Cell 1 PASS.** এখন R1-এর fold-aware TF-IDF foundation তৈরি হয়েছে।

Verified:

* **5 folds:** `[0,1,2,3,4]`
* **22,821 sessions** mapped to exactly one fold
* **6,139,854 turns** processed
* Fold-specific IDF built without target/label statistics
* `HashingVectorizer`: **262,144 features**, word 1–2 grams
* Full 6.1M-row count matrix materialize করা হয়নি
* Fold isolation preserved
* Serialization verified
* `R1_CELL_1_READY = True`

এখন **Cell 2** হবে **mathematical-token / math-pattern signal extraction**। এটা TF-IDF-এর বিকল্প নয়; sparse retrieval score-এর complementary signal হবে।

Cell 2-তে আমরা আগে diagnostic/build করব, যাতে numbers, fractions, operators, equations ইত্যাদি objective matching-এ useful signal হিসেবে পাওয়া যায়—কিন্তু target ব্যবহার করা হবে না।


In [14]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 1B — FOLD-AWARE WORD TF-IDF SCORING INTERFACE
# ==============================================================================
#
# PURPOSE
# -------
# Cell 1 built and serialized the fold-specific hashed TF-IDF IDF vectors.
#
# This cell exposes the scoring interface required by Cell 5:
#
#     r1_word_score_session(...)
#
# It does NOT refit TF-IDF.
# It does NOT use labels.
# It does NOT use target statistics.
#
# For a response in held-out fold F:
#
#     objective + session-local turns
#             ↓
#     HashingVectorizer
#             ↓
#     fold-F training-only IDF
#             ↓
#     row L2 normalization
#             ↓
#     cosine similarity
#
# ==============================================================================

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from scipy import sparse


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 1B — WORD TF-IDF SCORING INTERFACE")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY GATES
# ==============================================================================

assert R1_CELL_0_READY is True, (
    "R1 Cell 0 must pass."
)

assert R1_CELL_1_READY is True, (
    "R1 Cell 1 TF-IDF artifact build must pass."
)

assert R1_TFIDF_READY is True, (
    "R1 TF-IDF artifacts are not ready."
)

assert R1_FOLD_AWARE_READY is True, (
    "Fold-aware TF-IDF artifacts are not ready."
)

assert R1_TFIDF_TARGET_FREE is True, (
    "TF-IDF artifacts must remain target-free."
)


print("\nR1 Cell 0 dependency : PASS")
print("R1 Cell 1 dependency : PASS")


# ==============================================================================
# 1. CONFIGURATION CONTRACT
# ==============================================================================

R1_WORD_SCORER_FEATURES = int(
    WORD_N_FEATURES
)

R1_WORD_SCORER_NGRAM_RANGE = tuple(
    WORD_NGRAM_RANGE
)

R1_WORD_SCORER_SUBLINEAR_TF = bool(
    WORD_SUBLINEAR_TF
)


print("\n" + "=" * 80)
print("WORD TF-IDF SCORER CONFIGURATION")
print("=" * 80)

print(
    "Hash dimensions :",
    f"{R1_WORD_SCORER_FEATURES:,}",
)

print(
    "N-gram range    :",
    R1_WORD_SCORER_NGRAM_RANGE,
)

print(
    "Sublinear TF    :",
    R1_WORD_SCORER_SUBLINEAR_TF,
)

print(
    "Norm before IDF :",
    "None",
)

print(
    "Final similarity:",
    "L2-normalized cosine",
)


# ==============================================================================
# 2. RECREATE THE EXACT HASHING VECTORIZER
# ==============================================================================
#
# This MUST match Cell 1's representation contract.
#
# No vocabulary is fitted.
# HashingVectorizer is deterministic for the same configuration.
#
# ==============================================================================

word_vectorizer_r1 = HashingVectorizer(
    n_features=R1_WORD_SCORER_FEATURES,
    ngram_range=R1_WORD_SCORER_NGRAM_RANGE,
    lowercase=True,
    strip_accents="unicode",
    alternate_sign=False,
    norm=None,
    dtype=TFIDF_DTYPE,
)


# ==============================================================================
# 3. FOLD-SPECIFIC IDF CACHE
# ==============================================================================

word_idf_r1 = {}

word_tfidf_configs_r1 = {}

for heldout_fold in [0, 1, 2, 3, 4]:

    fold_dir = (
        R1_TFIDF_ROOT
        / f"fold_{heldout_fold}"
    )

    idf_path = (
        fold_dir
        / "word_idf.npy"
    )

    config_path = (
        fold_dir
        / "tfidf_config.json"
    )

    assert idf_path.exists(), (
        f"Missing fold-{heldout_fold} IDF artifact."
    )

    assert config_path.exists(), (
        f"Missing fold-{heldout_fold} TF-IDF config."
    )

    idf = np.load(
        idf_path,
        mmap_mode="r",
    )

    assert idf.shape == (
        R1_WORD_SCORER_FEATURES,
    ), (
        f"Fold {heldout_fold} IDF shape mismatch: "
        f"{idf.shape}"
    )

    assert np.isfinite(
        idf
    ).all(), (
        f"Fold {heldout_fold} IDF contains "
        "non-finite values."
    )

    assert (
        idf >= 1.0
    ).all(), (
        f"Fold {heldout_fold} IDF contains "
        "values below 1."
    )

    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as f:

        cfg = json.load(f)

    assert (
        int(cfg["heldout_fold"])
        == heldout_fold
    )

    assert (
        cfg["target_used"] is False
    )

    assert (
        cfg["target_statistics_used"] is False
    )

    assert (
        cfg["representation"]
        == "hashed_word_tfidf"
    )

    assert (
        int(cfg["n_features"])
        == R1_WORD_SCORER_FEATURES
    )

    assert tuple(
        cfg["ngram_range"]
    ) == R1_WORD_SCORER_NGRAM_RANGE

    word_idf_r1[
        heldout_fold
    ] = idf

    word_tfidf_configs_r1[
        heldout_fold
    ] = cfg


print(
    "\nFold-specific IDF artifacts loaded:",
    sorted(word_idf_r1.keys()),
)


# ==============================================================================
# 4. TEXT NORMALIZATION CONTRACT
# ==============================================================================
#
# Cell 1 uses canonical `text_norm`.
#
# Do not introduce a new normalization here.
# HashingVectorizer itself performs lowercase + unicode accent handling.
#
# Empty/null values become empty strings.
# ==============================================================================

def _r1_word_text(value):
    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return str(value)


# ==============================================================================
# 5. ROW-WISE L2 NORMALIZATION
# ==============================================================================

def _r1_l2_normalize_rows(
    X,
):
    """
    Row-wise L2 normalization for sparse matrices.

    X is expected to be CSR.
    """

    X = X.tocsr()

    row_norms = np.sqrt(
        np.asarray(
            X.multiply(X).sum(axis=1)
        ).ravel()
    )

    nonzero = (
        row_norms > 0
    )

    if np.any(nonzero):

        X = X.tolil(
            copy=False
        )

        X[nonzero] = (
            X[nonzero].multiply(
                1.0
                /
                row_norms[
                    nonzero
                ][:, None]
            )
        )

        X = X.tocsr()

    return X


# ==============================================================================
# 6. TF → IDF → L2 NORMALIZATION
# ==============================================================================

def _r1_hash_tfidf(
    texts,
    heldout_fold,
):
    """
    Convert texts to fold-specific hashed TF-IDF vectors.

    IMPORTANT:
        IDF is loaded from the artifact corresponding
        to the held-out fold.

    No fitting occurs here.
    """

    heldout_fold = int(
        heldout_fold
    )

    assert heldout_fold in word_idf_r1, (
        f"No fold-specific IDF loaded for fold "
        f"{heldout_fold}."
    )

    clean_texts = [
        _r1_word_text(x)
        for x in texts
    ]

    X_tf = (
        word_vectorizer_r1.transform(
            clean_texts
        )
    ).tocsr()

    assert X_tf.shape[1] == (
        R1_WORD_SCORER_FEATURES
    )

    idf = word_idf_r1[
        heldout_fold
    ]

    # --------------------------------------------------------------
    # TF × IDF
    # --------------------------------------------------------------

    X_tfidf = (
        X_tf.multiply(
            idf
        )
    ).tocsr()

    # --------------------------------------------------------------
    # L2 normalize
    # --------------------------------------------------------------

    X_tfidf = _r1_l2_normalize_rows(
        X_tfidf
    )

    return X_tfidf


# ==============================================================================
# 7. OBJECTIVE TEXT LOOKUP
# ==============================================================================
#
# Cell 5 already loads objective_char etc.
# We intentionally build an independent objective text lookup so
# this scorer has an explicit and auditable dependency.
#
# ==============================================================================

assert (
    "objective_catalogue_r1" in globals()
), (
    "Cell 5 objective catalogue is not loaded. "
    "Run Cell 5 setup before smoke-testing the scorer."
)

assert (
    "objective_uid" in objective_catalogue_r1.columns
)

assert (
    "objective_raw" in objective_catalogue_r1.columns
)

objective_word_text_r1 = (
    objective_catalogue_r1[
        [
            "objective_uid",
            "objective_raw",
        ]
    ]
    .drop_duplicates(
        subset=[
            "objective_uid"
        ]
    )
    .set_index(
        "objective_uid"
    )[
        "objective_raw"
    ]
    .map(_r1_word_text)
    .to_dict()
)


# ==============================================================================
# 8. FOLD-AWARE WORD SCORE FUNCTION
# ==============================================================================

def r1_word_score_session(
    response_row,
    turns_local,
):
    """
    Fold-aware word TF-IDF cosine similarity.

    Parameters
    ----------
    response_row : pandas.Series
        Must contain:
            response_id
            session_id
            objective_uid
            fold

    turns_local : pandas.DataFrame
        Session-local canonical turns.

    Returns
    -------
    numpy.ndarray
        Float32 cosine similarity for every row in turns_local.

    Leakage contract
    ----------------
    For a response in fold F:

        objective
             +
        session-local turns
             ↓
        fold-F IDF

    The fold-F IDF was built from training folds only.
    """

    assert (
        "objective_uid" in response_row.index
    )

    assert (
        "fold" in response_row.index
    )

    objective_uid = (
        response_row[
            "objective_uid"
        ]
    )

    heldout_fold = int(
        response_row[
            "fold"
        ]
    )

    assert heldout_fold in {
        0,
        1,
        2,
        3,
        4,
    }

    assert (
        objective_uid
        in objective_word_text_r1
    ), (
        f"Unknown objective_uid: "
        f"{objective_uid}"
    )

    objective_text = (
        objective_word_text_r1[
            objective_uid
        ]
    )

    turn_texts = (
        turns_local[
            "text_norm"
        ]
        .map(_r1_word_text)
        .tolist()
    )

    # --------------------------------------------------------------
    # Transform objective + turns together.
    #
    # Hashing is deterministic; no fitting occurs.
    # --------------------------------------------------------------

    texts = [
        objective_text,
        *turn_texts,
    ]

    X = _r1_hash_tfidf(
        texts=texts,
        heldout_fold=heldout_fold,
    )

    objective_vector = (
        X[0]
    )

    turn_matrix = (
        X[1:]
    )

    # --------------------------------------------------------------
    # Cosine similarity after L2 normalization.
    # --------------------------------------------------------------

    scores = (
        turn_matrix
        @
        objective_vector.T
    ).toarray().ravel()

    scores = np.asarray(
        scores,
        dtype=np.float32,
    )

    # Numerical safety.
    scores = np.clip(
        scores,
        -1.0,
        1.0,
    )

    return scores


# ==============================================================================
# 9. PUBLIC SCORER CONTRACT
# ==============================================================================

assert callable(
    r1_word_score_session
)

R1_WORD_SCORER_READY = True


# ==============================================================================
# 10. SMOKE TEST
# ==============================================================================

print("\n" + "=" * 80)
print("R1 WORD TF-IDF SCORER SMOKE TEST")
print("=" * 80)

assert (
    "session_turn_groups" in globals()
), (
    "Cell 5 session index is not available."
)

assert (
    len(session_turn_groups) > 0
), (
    "Session turn index is empty."
)

assert (
    len(retrieval_queries_r1) > 0
), (
    "Retrieval query table is empty."
)


smoke_response = (
    retrieval_queries_r1
    .iloc[0]
)

smoke_session_id = (
    smoke_response[
        "session_id"
    ]
)

smoke_turns = (
    session_turn_groups[
        smoke_session_id
    ]
)

smoke_scores = (
    r1_word_score_session(
        response_row=smoke_response,
        turns_local=smoke_turns,
    )
)

assert isinstance(
    smoke_scores,
    np.ndarray,
)

assert smoke_scores.dtype == (
    np.float32
)

assert len(
    smoke_scores
) == len(
    smoke_turns
)

assert np.isfinite(
    smoke_scores
).all()

assert (
    np.max(smoke_scores) <= 1.000001
)

assert (
    np.min(smoke_scores) >= -1.000001
)


print(
    "Smoke response_id :",
    smoke_response[
        "response_id"
    ],
)

print(
    "Smoke session_id  :",
    smoke_session_id,
)

print(
    "Held-out fold      :",
    int(
        smoke_response[
            "fold"
        ]
    ),
)

print(
    "Session-local turns:",
    f"{len(smoke_turns):,}",
)

print(
    "Score min          :",
    float(
        smoke_scores.min()
    ),
)

print(
    "Score max          :",
    float(
        smoke_scores.max()
    ),
)

print(
    "Score mean         :",
    float(
        smoke_scores.mean()
    ),
)


# ==============================================================================
# 11. FOLD-ISOLATION SELF-TEST
# ==============================================================================

print("\n" + "=" * 80)
print("FOLD-ISOLATION SELF-TEST")
print("=" * 80)

for f in [0, 1, 2, 3, 4]:

    cfg = word_tfidf_configs_r1[
        f
    ]

    training_folds = set(
        int(x)
        for x in cfg[
            "training_folds"
        ]
    )

    assert f not in (
        training_folds
    ), (
        f"Fold {f} appears in its own "
        "training_folds."
    )

    assert training_folds == (
        set([0, 1, 2, 3, 4])
        - {f}
    )

    assert (
        cfg[
            "target_used"
        ] is False
    )

    assert (
        cfg[
            "target_statistics_used"
        ] is False
    )

print(
    "Fold isolation checks : 5/5 PASS"
)


# ==============================================================================
# 12. FINAL STATUS
# ==============================================================================

R1_WORD_SCORER_TARGET_FREE = True
R1_WORD_SCORER_FOLD_AWARE = True
R1_WORD_SCORER_SERIALIZATION_READY = True

R1_WORD_SCORER_READY = (
    R1_WORD_SCORER_READY
    and R1_WORD_SCORER_TARGET_FREE
    and R1_WORD_SCORER_FOLD_AWARE
    and R1_WORD_SCORER_SERIALIZATION_READY
)

print("\n" + "=" * 80)
print("R1 CELL 1B STATUS")
print("=" * 80)

print(
    "Word scorer ready       :",
    R1_WORD_SCORER_READY,
)

print(
    "Fold-aware              :",
    R1_WORD_SCORER_FOLD_AWARE,
)

print(
    "Target-free             :",
    R1_WORD_SCORER_TARGET_FREE,
)

assert R1_WORD_SCORER_READY is True

print(
    "\nR1_CELL_1B_READY        :",
    R1_WORD_SCORER_READY,
)

print("=" * 80)
print(
    "R1 CELL 1B — WORD TF-IDF SCORER: PASS"
)
print("=" * 80)

gc.collect()

R1_CELL_1B_READY = True
R1_WORD_SCORER_READY = True

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 1B — WORD TF-IDF SCORING INTERFACE

R1 Cell 0 dependency : PASS
R1 Cell 1 dependency : PASS

WORD TF-IDF SCORER CONFIGURATION
Hash dimensions : 262,144
N-gram range    : (1, 2)
Sublinear TF    : True
Norm before IDF : None
Final similarity: L2-normalized cosine

Fold-specific IDF artifacts loaded: [0, 1, 2, 3, 4]

R1 WORD TF-IDF SCORER SMOKE TEST
Smoke response_id : aaaavsh
Smoke session_id  : bcaufvc
Held-out fold      : 3
Session-local turns: 330
Score min          : 0.0
Score max          : 0.9344601631164551
Score mean         : 0.01561779249459505

FOLD-ISOLATION SELF-TEST
Fold isolation checks : 5/5 PASS

R1 CELL 1B STATUS
Word scorer ready       : True
Fold-aware              : True
Target-free             : True

R1_CELL_1B_READY        : True
R1 CELL 1B — WORD TF-IDF SCORER: PASS


In [3]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 2 — MATHEMATICAL TOKEN EXTRACTION
# ==============================================================================

import re
import gc
import json
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 2 — MATHEMATICAL TOKEN EXTRACTION")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY GATE
# ==============================================================================

assert R1_CELL_0_READY is True, (
    "R1 Cell 0 verification must pass before Cell 2."
)

assert R1_CELL_1_READY is True, (
    "R1 Cell 1 TF-IDF build must pass before Cell 2."
)

print("\nR1 Cell 0 dependency : PASS")
print("R1 Cell 1 dependency : PASS")


# ==============================================================================
# 1. OUTPUT PATHS
# ==============================================================================

R1_MATH_ROOT = (
    R1_ROOT
    / "math_artifacts"
)

R1_MATH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R1_TURN_MATH_PATH = (
    R1_MATH_ROOT
    / "turn_math_tokens.parquet"
)

R1_OBJECTIVE_MATH_PATH = (
    R1_MATH_ROOT
    / "objective_math_tokens.parquet"
)

R1_MATH_MANIFEST_PATH = (
    R1_MATH_ROOT
    / "math_manifest.json"
)


print("\n" + "=" * 80)
print("R1 MATH ARTIFACT PATHS")
print("=" * 80)

print(
    "Turn math tokens      :",
    R1_TURN_MATH_PATH,
)

print(
    "Objective math tokens :",
    R1_OBJECTIVE_MATH_PATH,
)

print(
    "Math manifest         :",
    R1_MATH_MANIFEST_PATH,
)


# ==============================================================================
# 2. MATHEMATICAL TOKEN CONTRACT
#
# We intentionally keep this signal narrow.
#
# Included:
#   - integers
#   - decimals
#   - fractions
#   - percentages
#   - arithmetic operators
#   - comparison/equality operators
#   - common mathematical symbols
#   - common measurement units
#   - common number words
#
# This is NOT a replacement for lexical TF-IDF.
# It is a complementary exact-ish mathematical-content signal.
# ==============================================================================


# --------------------------------------------------------------------------
# Numeric expressions
# --------------------------------------------------------------------------

FRACTION_RE = re.compile(
    r"(?<![\w])"
    r"\d+(?:\.\d+)?"
    r"\s*/\s*"
    r"\d+(?:\.\d+)?"
    r"(?![\w])"
)

PERCENT_RE = re.compile(
    r"(?<![\w])"
    r"\d+(?:\.\d+)?\s*%"
    r"(?![\w])"
)

DECIMAL_RE = re.compile(
    r"(?<![\w])"
    r"\d+\.\d+"
    r"(?![\w])"
)

INTEGER_RE = re.compile(
    r"(?<![\w])"
    r"\d+(?![\w])"
)


# --------------------------------------------------------------------------
# Mathematical operators / symbols
# --------------------------------------------------------------------------

MATH_SYMBOL_RE = re.compile(
    r"[+\-×÷*/=<>≤≥≠≈±]"
)


# --------------------------------------------------------------------------
# Units frequently occurring in elementary mathematics.
# Keep this intentionally conservative.
# --------------------------------------------------------------------------

UNIT_RE = re.compile(
    r"(?<![\w])"
    r"(?:"
    r"cm|mm|m|km|"
    r"g|kg|mg|"
    r"ml|l|"
    r"°|deg|"
    r"hours?|hrs?|minutes?|mins?|seconds?|secs?"
    r")"
    r"(?![\w])",
    flags=re.IGNORECASE,
)


# --------------------------------------------------------------------------
# Common number words.
# These are useful for objectives such as:
# "two digit", "three quarters", etc.
# --------------------------------------------------------------------------

NUMBER_WORDS = {
    "zero",
    "one",
    "two",
    "three",
    "four",
    "five",
    "six",
    "seven",
    "eight",
    "nine",
    "ten",
    "eleven",
    "twelve",
    "thirteen",
    "fourteen",
    "fifteen",
    "sixteen",
    "seventeen",
    "eighteen",
    "nineteen",
    "twenty",
    "thirty",
    "forty",
    "fifty",
    "sixty",
    "seventy",
    "eighty",
    "ninety",
    "hundred",
    "thousand",
    "million",
    "half",
    "halves",
    "quarter",
    "quarters",
    "third",
    "thirds",
    "fourth",
    "fourths",
}


WORD_RE = re.compile(
    r"\b[a-z]+\b",
    flags=re.IGNORECASE,
)


# ==============================================================================
# 3. TOKEN NORMALIZATION
# ==============================================================================

def _normalize_numeric_token(token):
    """
    Normalize numeric expressions into stable compact forms.

    Examples:
        1,000     -> 1000
        25 %      -> 25%
        1 / 2     -> 1/2
        3.50      -> 3.50
    """

    token = str(token).strip().lower()

    token = token.replace(
        ",",
        "",
    )

    token = re.sub(
        r"\s+",
        "",
        token,
    )

    return token


def extract_math_tokens(text):
    """
    Extract a deterministic mathematical signature.

    Returns:
        tuple[str, ...]
    """

    if text is None:
        return tuple()

    text = str(text)

    if not text.strip():
        return tuple()

    text_lower = text.lower()

    tokens = []

    # ------------------------------------------------------------------
    # Fractions
    # ------------------------------------------------------------------

    for match in FRACTION_RE.finditer(
        text_lower
    ):
        tokens.append(
            "FRAC:" +
            _normalize_numeric_token(
                match.group(0)
            )
        )

    # ------------------------------------------------------------------
    # Percentages
    # ------------------------------------------------------------------

    for match in PERCENT_RE.finditer(
        text_lower
    ):
        tokens.append(
            "PCT:" +
            _normalize_numeric_token(
                match.group(0)
            )
        )

    # ------------------------------------------------------------------
    # Decimals
    #
    # Avoid duplicating decimal numbers as INTEGER tokens.
    # ------------------------------------------------------------------

    decimal_spans = []

    for match in DECIMAL_RE.finditer(
        text_lower
    ):
        decimal_spans.append(
            match.span()
        )

        tokens.append(
            "NUM:" +
            _normalize_numeric_token(
                match.group(0)
            )
        )

    # ------------------------------------------------------------------
    # Integers
    # ------------------------------------------------------------------

    for match in INTEGER_RE.finditer(
        text_lower
    ):
        start, end = match.span()

        inside_decimal = any(
            ds <= start and end <= de
            for ds, de in decimal_spans
        )

        if not inside_decimal:
            tokens.append(
                "NUM:" +
                _normalize_numeric_token(
                    match.group(0)
                )
            )

    # ------------------------------------------------------------------
    # Mathematical symbols
    # ------------------------------------------------------------------

    for match in MATH_SYMBOL_RE.finditer(
        text_lower
    ):
        tokens.append(
            "SYM:" +
            match.group(0)
        )

    # ------------------------------------------------------------------
    # Units
    # ------------------------------------------------------------------

    for match in UNIT_RE.finditer(
        text_lower
    ):
        unit = match.group(0).lower()

        unit_aliases = {
            "mins": "min",
            "minute": "min",
            "minutes": "min",
            "hrs": "hr",
            "hour": "hr",
            "hours": "hr",
            "secs": "sec",
            "second": "sec",
            "seconds": "sec",
        }

        unit = unit_aliases.get(
            unit,
            unit,
        )

        tokens.append(
            "UNIT:" + unit
        )

    # ------------------------------------------------------------------
    # Number words
    # ------------------------------------------------------------------

    for word in WORD_RE.findall(
        text_lower
    ):
        word = word.lower()

        if word in NUMBER_WORDS:
            tokens.append(
                "WORD:" + word
            )

    # ------------------------------------------------------------------
    # Deterministic unique signature.
    #
    # Sorting makes the representation order-independent.
    # This is deliberate because the downstream feature is overlap,
    # not mathematical sequence reconstruction.
    # ------------------------------------------------------------------

    return tuple(
        sorted(
            set(tokens)
        )
    )


# ==============================================================================
# 4. UNIT TESTS
# ==============================================================================

print("\n" + "=" * 80)
print("MATH TOKEN SELF-TEST")
print("=" * 80)


math_test_cases = {
    "What is 3 + 4?": {
        "NUM:3",
        "NUM:4",
        "SYM:+",
    },

    "What is 1/2 of 20?": {
        "FRAC:1/2",
        "NUM:20",
    },

    "25% of 200": {
        "PCT:25%",
        "NUM:200",
    },

    "3.5 kg is how many grams?": {
        "NUM:3.5",
        "UNIT:kg",
    },

    "Compare 7 > 5": {
        "NUM:7",
        "NUM:5",
        "SYM:>",
    },

    "two thirds": {
        "WORD:two",
        "WORD:thirds",
    },
}


for text, expected_subset in math_test_cases.items():

    observed = set(
        extract_math_tokens(text)
    )

    missing = (
        expected_subset
        - observed
    )

    assert not missing, (
        f"Math token self-test failed for: {text!r} | "
        f"missing={missing} | observed={observed}"
    )


print(
    f"Self-tests passed: "
    f"{len(math_test_cases)}/{len(math_test_cases)}"
)


# ==============================================================================
# 5. OBJECTIVE MATH SIGNATURES
# ==============================================================================

print("\n" + "=" * 80)
print("BUILD OBJECTIVE MATH SIGNATURES")
print("=" * 80)


objective_math_rows = []

for row in objective_catalogue[
    [
        "objective_uid",
        "objective_raw",
    ]
].itertuples(
    index=False
):

    tokens = extract_math_tokens(
        row.objective_raw
    )

    objective_math_rows.append(
        {
            "objective_uid": row.objective_uid,
            "math_tokens": " ".join(tokens),
            "n_math_tokens": len(tokens),
        }
    )


objective_math = pd.DataFrame(
    objective_math_rows
)


assert len(objective_math) == 398

assert objective_math[
    "objective_uid"
].is_unique

assert objective_math[
    "objective_uid"
].notna().all()


objective_math.to_parquet(
    R1_OBJECTIVE_MATH_PATH,
    index=False,
)


print(
    "Objective rows:",
    f"{len(objective_math):,}",
)

print(
    "Objectives with math tokens:",
    int(
        (
            objective_math["n_math_tokens"]
            > 0
        ).sum()
    ),
)

print(
    "Objective artifact:",
    R1_OBJECTIVE_MATH_PATH,
)


# ==============================================================================
# 6. STREAM TURN MATH SIGNATURES
#
# We preserve:
#     session_id
#     turn_uid
#     turn_index
#     math_tokens
#     n_math_tokens
#
# No target.
# No response label.
# No fold.
#
# Fold is deliberately not copied here because the frozen session→fold
# mapping remains available separately through R0 retrieval_queries.
# ==============================================================================


print("\n" + "=" * 80)
print("STREAM TURN MATH SIGNATURES")
print("=" * 80)


TURN_BATCH_SIZE_MATH = 50_000

turn_pf_math = pq.ParquetFile(
    R0_TURN_SOURCE
)


math_schema = pa.schema(
    [
        pa.field(
            "session_id",
            pa.string(),
        ),
        pa.field(
            "turn_uid",
            pa.string(),
        ),
        pa.field(
            "turn_index",
            pa.int32(),
        ),
        pa.field(
            "math_tokens",
            pa.string(),
        ),
        pa.field(
            "n_math_tokens",
            pa.int16(),
        ),
    ]
)


writer = pq.ParquetWriter(
    R1_TURN_MATH_PATH,
    math_schema,
    compression="zstd",
)


total_written = 0
total_with_math = 0
total_math_tokens = 0

math_token_frequency = Counter()


try:

    for batch_no, batch in enumerate(
        turn_pf_math.iter_batches(
            batch_size=TURN_BATCH_SIZE_MATH,
            columns=[
                "session_id",
                "turn_uid",
                "turn_index",
                "text_norm",
            ],
        ),
        start=1,
    ):

        batch_df = batch.to_pandas()

        # --------------------------------------------------------------
        # Extract signatures
        # --------------------------------------------------------------

        signatures = []

        for text in batch_df[
            "text_norm"
        ]:

            tokens = extract_math_tokens(
                text
            )

            signatures.append(
                tokens
            )

        # --------------------------------------------------------------
        # Compact string representation
        # --------------------------------------------------------------

        math_token_strings = [
            " ".join(tokens)
            for tokens in signatures
        ]

        token_counts = np.asarray(
            [
                len(tokens)
                for tokens in signatures
            ],
            dtype=np.int16,
        )

        # --------------------------------------------------------------
        # Batch statistics
        # --------------------------------------------------------------

        n_with_math = int(
            np.count_nonzero(
                token_counts
            )
        )

        total_written += len(
            batch_df
        )

        total_with_math += (
            n_with_math
        )

        total_math_tokens += int(
            token_counts.sum()
        )

        for tokens in signatures:
            math_token_frequency.update(
                tokens
            )

        # --------------------------------------------------------------
        # Arrow batch
        # --------------------------------------------------------------

        out_df = pd.DataFrame(
            {
                "session_id": (
                    batch_df[
                        "session_id"
                    ].astype(str)
                ),
                "turn_uid": (
                    batch_df[
                        "turn_uid"
                    ].astype(str)
                ),
                "turn_index": (
                    batch_df[
                        "turn_index"
                    ].astype(np.int32)
                ),
                "math_tokens": (
                    math_token_strings
                ),
                "n_math_tokens": (
                    token_counts
                ),
            }
        )

        table = pa.Table.from_pandas(
            out_df,
            schema=math_schema,
            preserve_index=False,
        )

        writer.write_table(
            table
        )

        # --------------------------------------------------------------
        # Progress
        # --------------------------------------------------------------

        if batch_no % 20 == 0:
            print(
                f"  batches={batch_no:,} | "
                f"rows={total_written:,} | "
                f"rows_with_math={total_with_math:,}"
            )

        del batch_df
        del signatures
        del math_token_strings
        del token_counts
        del out_df
        del table

        gc.collect()

finally:
    writer.close()


# ==============================================================================
# 7. TURN ARTIFACT RELOAD + CONTRACT
# ==============================================================================

print("\n" + "=" * 80)
print("TURN MATH ARTIFACT AUDIT")
print("=" * 80)


assert R1_TURN_MATH_PATH.exists(), (
    "Turn math-token artifact was not written."
)


turn_math_pf = pq.ParquetFile(
    R1_TURN_MATH_PATH
)

turn_math_rows = int(
    turn_math_pf.metadata.num_rows
)

print(
    "Rows written :",
    f"{turn_math_rows:,}",
)

print(
    "Expected     :",
    f"{turn_total_rows:,}",
)

assert turn_math_rows == turn_total_rows, (
    "Turn math artifact population changed."
)


turn_math_sample = pd.read_parquet(
    R1_TURN_MATH_PATH
)


assert list(
    turn_math_sample.columns
) == [
    "session_id",
    "turn_uid",
    "turn_index",
    "math_tokens",
    "n_math_tokens",
]


assert (
    turn_math_sample[
        "turn_uid"
    ].is_unique
), (
    "turn_uid is not unique in math artifact sample."
)

assert (
    turn_math_sample[
        "session_id"
    ].notna().all()
)

assert (
    turn_math_sample[
        "turn_uid"
    ].notna().all()
)


# ==============================================================================
# 8. FULL IDENTITY CHECK
#
# We use Parquet metadata / DuckDB-style pandas-free check only through
# streaming batches so the entire 6.1M-row table is never loaded.
# ==============================================================================

seen_turns = set()

# Memory-safe identity check:
# use Arrow batches and only retain IDs temporarily.
#
# This set is large but manageable (~6.1M strings) and is used only
# for this audit. It is released immediately afterward.

duplicate_turn_uid_count = 0
stream_rows_checked = 0

for batch in turn_math_pf.iter_batches(
    batch_size=TURN_BATCH_SIZE_MATH,
    columns=[
        "turn_uid",
    ],
):

    ids = (
        batch.column(
            "turn_uid"
        ).to_pylist()
    )

    for turn_uid in ids:

        if turn_uid in seen_turns:
            duplicate_turn_uid_count += 1
        else:
            seen_turns.add(
                turn_uid
            )

    stream_rows_checked += len(
        ids
    )


assert stream_rows_checked == turn_total_rows

assert duplicate_turn_uid_count == 0, (
    "Duplicate turn_uid detected in math artifact."
)

assert len(seen_turns) == turn_total_rows, (
    "Math artifact turn identity population changed."
)


del seen_turns
gc.collect()


# ==============================================================================
# 9. MATH COVERAGE DIAGNOSTICS
# ==============================================================================

math_rows_without_tokens = (
    turn_math_rows
    - total_with_math
)

math_coverage = (
    total_with_math
    / turn_math_rows
    if turn_math_rows
    else 0.0
)


print("\n" + "=" * 80)
print("MATH TOKEN COVERAGE")
print("=" * 80)

print(
    "Turn rows:",
    f"{turn_math_rows:,}",
)

print(
    "Rows with math tokens:",
    f"{total_with_math:,}",
)

print(
    "Rows without math tokens:",
    f"{math_rows_without_tokens:,}",
)

print(
    "Math-token row coverage:",
    f"{math_coverage:.4%}",
)

print(
    "Total extracted math tokens:",
    f"{total_math_tokens:,}",
)


# ==============================================================================
# 10. TOP TOKEN DIAGNOSTICS
# ==============================================================================

print("\n" + "=" * 80)
print("TOP MATHEMATICAL TOKENS")
print("=" * 80)

top_math_tokens = (
    math_token_frequency
    .most_common(30)
)

for token, count in top_math_tokens:
    print(
        f"{token:24s} {count:,}"
    )


# ==============================================================================
# 11. TARGET / LABEL ISOLATION
# ==============================================================================

math_artifact_columns = set(
    pq.read_schema(
        R1_TURN_MATH_PATH
    ).names
)

objective_math_columns = set(
    pq.read_schema(
        R1_OBJECTIVE_MATH_PATH
    ).names
)


assert not (
    math_artifact_columns
    & PROHIBITED_R1_INPUT_FIELDS
), (
    "Prohibited target/evaluation field found "
    "in turn math artifact."
)

assert not (
    objective_math_columns
    & PROHIBITED_R1_INPUT_FIELDS
), (
    "Prohibited target/evaluation field found "
    "in objective math artifact."
)


# ==============================================================================
# 12. MANIFEST
# ==============================================================================

math_manifest = {
    "stage": "R1_SPARSE_RETRIEVAL",
    "component": "mathematical_token_extraction",

    "turn_artifact": str(
        R1_TURN_MATH_PATH
    ),

    "objective_artifact": str(
        R1_OBJECTIVE_MATH_PATH
    ),

    "turn_rows": int(
        turn_math_rows
    ),

    "objective_rows": int(
        len(objective_math)
    ),

    "turn_rows_with_math_tokens": int(
        total_with_math
    ),

    "turn_math_row_coverage": float(
        math_coverage
    ),

    "total_math_tokens": int(
        total_math_tokens
    ),

    "token_families": [
        "numeric",
        "fraction",
        "percentage",
        "operator",
        "comparison",
        "unit",
        "number_word",
    ],

    "target_used": False,
    "target_statistics_used": False,
    "label_used": False,

    "canonical_turn_source": str(
        R0_TURN_SOURCE
    ),

    "canonical_source_modified": False,

    "deterministic": True,

    "full_turn_table_materialized": False,
}


with open(
    R1_MATH_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        math_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


assert R1_MATH_MANIFEST_PATH.exists()


# ==============================================================================
# 13. FINAL CELL 2 FLAGS
# ==============================================================================

R1_MATH_EXTRACTION_READY = True
R1_MATH_IDENTITY_READY = True
R1_MATH_TARGET_FREE = True
R1_MATH_SERIALIZATION_READY = True

R1_CELL_2_READY = (
    R1_MATH_EXTRACTION_READY
    and R1_MATH_IDENTITY_READY
    and R1_MATH_TARGET_FREE
    and R1_MATH_SERIALIZATION_READY
)


# ==============================================================================
# 14. FINAL STATUS
# ==============================================================================

print("\n" + "=" * 80)
print("R1 CELL 2 STATUS")
print("=" * 80)

print(
    "Turn math extraction ready :",
    R1_MATH_EXTRACTION_READY,
)

print(
    "Identity contract ready    :",
    R1_MATH_IDENTITY_READY,
)

print(
    "Target-free                :",
    R1_MATH_TARGET_FREE,
)

print(
    "Serialization ready        :",
    R1_MATH_SERIALIZATION_READY,
)

print(
    "R1_CELL_2_READY            :",
    R1_CELL_2_READY,
)

assert R1_CELL_2_READY is True

print("=" * 80)
print("R1 CELL 2 — MATHEMATICAL TOKEN EXTRACTION: PASS")
print("=" * 80)


# Release audit objects.
del turn_math_sample
del objective_math
del math_token_frequency
gc.collect()

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 2 — MATHEMATICAL TOKEN EXTRACTION

R1 Cell 0 dependency : PASS
R1 Cell 1 dependency : PASS

R1 MATH ARTIFACT PATHS
Turn math tokens      : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\math_artifacts\turn_math_tokens.parquet
Objective math tokens : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\math_artifacts\objective_math_tokens.parquet
Math manifest         : d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\math_artifacts\math_manifest.json

MATH TOKEN SELF-TEST
Self-tests passed: 6/6

BUILD OBJECTIVE MATH SIGNATURES
Objective rows: 398
Objectives with math tokens: 101
Objective artifact: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\math_artifacts\objective_math_tokens.parquet

STREAM TURN MATH SIGNATURES
  batches=20 | rows=1,000,000 | rows_with_math=453,693
  batches=40 | rows=2,000,000 | rows_with

33

In [5]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 4 — CHARACTER SIMILARITY
# ==============================================================================

import gc
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 4 — CHARACTER SIMILARITY")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY GATE
# ==============================================================================

assert R1_CELL_2_READY is True, (
    "R1 Cell 2 must pass before Cell 4."
)

assert R1_MATH_EXTRACTION_READY is True, (
    "Mathematical token extraction must be ready."
)

assert R1_MATH_TARGET_FREE is True, (
    "Math artifact must remain target-free."
)

print("\nR1 Cell 2 dependency : PASS")
print("Math artifact        : PASS")
print("Target isolation     : PASS")


# ==============================================================================
# 1. CHARACTER ARTIFACT ROOT
# ==============================================================================

R1_CHAR_ROOT = (
    R1_ROOT / "char_artifacts"
)

R1_CHAR_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R1_CHAR_MANIFEST_PATH = (
    R1_CHAR_ROOT / "char_manifest.json"
)

print("\n" + "=" * 80)
print("CHARACTER ARTIFACT ROOT")
print("=" * 80)

print("Root:", R1_CHAR_ROOT)


# ==============================================================================
# 2. CHARACTER NORMALIZATION
# ==============================================================================

def normalize_char_text(text):
    """
    Conservative character-retrieval normalization.

    Preserve:
      - digits
      - operators
      - fraction separators
      - lexical fragments
      - punctuation useful for character n-grams

    Normalize only:
      - case
      - whitespace
    """

    if text is None:
        return ""

    text = str(text).lower()

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


# ==============================================================================
# 3. SELF-TEST
# ==============================================================================

print("\n" + "=" * 80)
print("CHARACTER NORMALIZATION SELF-TEST")
print("=" * 80)

char_tests = {
    "Hello   World": "hello world",
    "1 / 2": "1 / 2",
    "3+4 = 7": "3+4 = 7",
    "  Fraction\tQuestion  ": "fraction question",
}

for raw, expected in char_tests.items():

    observed = normalize_char_text(raw)

    assert observed == expected, (
        f"Normalization mismatch: "
        f"{raw!r} -> {observed!r}, "
        f"expected {expected!r}"
    )

print(
    f"Self-tests passed: "
    f"{len(char_tests)}/{len(char_tests)}"
)


# ==============================================================================
# 4. LOAD R0 OBJECTIVE CATALOGUE
#
# IMPORTANT:
# R0 deliberately exposes only:
#
#     objective_uid
#     objective_raw
#
# Therefore R1 derives its retrieval-specific normalized text here.
# R0 remains immutable.
# ==============================================================================

OBJECTIVE_UID_COL = "objective_uid"
OBJECTIVE_TEXT_COL = "objective_raw"

assert OBJECTIVE_UID_COL in objective_catalogue.columns, (
    "R0 objective catalogue is missing objective_uid."
)

assert OBJECTIVE_TEXT_COL in objective_catalogue.columns, (
    "R0 objective catalogue is missing objective_raw."
)

objective_char = (
    objective_catalogue[
        [
            OBJECTIVE_UID_COL,
            OBJECTIVE_TEXT_COL,
        ]
    ]
    .copy()
)

objective_char["char_text"] = (
    objective_char[OBJECTIVE_TEXT_COL]
    .map(normalize_char_text)
)

assert objective_char[
    OBJECTIVE_UID_COL
].is_unique, (
    "objective_uid is not unique."
)

assert objective_char[
    OBJECTIVE_TEXT_COL
].notna().all(), (
    "objective_raw contains null values."
)

assert objective_char[
    "char_text"
].str.len().gt(0).all(), (
    "One or more objectives became empty after "
    "character normalization."
)

print("\n" + "=" * 80)
print("OBJECTIVE CHARACTER CORPUS")
print("=" * 80)

print(
    "Objectives:",
    f"{len(objective_char):,}",
)

print(
    "Empty normalized objectives:",
    int(
        (
            objective_char["char_text"]
            .str.len()
            == 0
        ).sum()
    ),
)


# ==============================================================================
# 5. LOAD CANONICAL TURN SOURCE
# ==============================================================================

TURN_SOURCE_PATH = R0_TURN_SOURCE

turn_pf_char = pq.ParquetFile(
    TURN_SOURCE_PATH
)

print("\n" + "=" * 80)
print("CANONICAL TURN SOURCE")
print("=" * 80)

print(
    "Turn source:",
    TURN_SOURCE_PATH,
)

print(
    "Turn rows:",
    f"{turn_pf_char.metadata.num_rows:,}",
)


# ==============================================================================
# 6. CHARACTER TF-IDF CONFIG
# ==============================================================================

CHAR_NGRAM_RANGE = (3, 5)
CHAR_MIN_DF = 2
CHAR_MAX_FEATURES = 500_000
CHAR_SUBLINEAR_TF = True

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=CHAR_MIN_DF,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=CHAR_SUBLINEAR_TF,
    lowercase=False,
    norm="l2",
    dtype=np.float32,
)


# ==============================================================================
# 7. STREAMING TURN DOCUMENT GENERATOR
# ==============================================================================

def normalized_turn_document_generator(
    parquet_file,
    batch_size=50_000,
):
    """
    Stream canonical turn text without materializing
    all 6.1M turns in Python memory.
    """

    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=["text_norm"],
    ):

        values = batch.column(
            "text_norm"
        ).to_pylist()

        for text in values:
            yield normalize_char_text(text)


# ==============================================================================
# 8. FIT CHARACTER TF-IDF
# ==============================================================================

print("\n" + "=" * 80)
print("FIT CHARACTER TF-IDF")
print("=" * 80)

print(
    "n-gram range :",
    CHAR_NGRAM_RANGE,
)

print(
    "min_df       :",
    CHAR_MIN_DF,
)

print(
    "max_features :",
    CHAR_MAX_FEATURES,
)

print(
    "\nFitting against canonical turn corpus..."
)

char_vectorizer.fit(
    normalized_turn_document_generator(
        turn_pf_char
    )
)

vocabulary_size = len(
    char_vectorizer.vocabulary_
)

print(
    "\nCharacter vocabulary:",
    f"{vocabulary_size:,}",
)

assert vocabulary_size > 0, (
    "Character TF-IDF vocabulary is empty."
)


# ==============================================================================
# 9. OBJECTIVE CHARACTER VECTORS
# ==============================================================================

objective_char_matrix = (
    char_vectorizer.transform(
        objective_char["char_text"]
    )
)

assert (
    objective_char_matrix.shape[0]
    == len(objective_char)
)

assert (
    objective_char_matrix.shape[1]
    == vocabulary_size
)

print("\n" + "=" * 80)
print("OBJECTIVE CHARACTER MATRIX")
print("=" * 80)

print(
    "Shape:",
    objective_char_matrix.shape,
)

print(
    "Non-zero entries:",
    objective_char_matrix.nnz,
)


# ==============================================================================
# 10. SERIALIZE CHARACTER VECTOR SPACE
# ==============================================================================

CHAR_VECTORIZER_PATH = (
    R1_CHAR_ROOT
    / "character_tfidf_vectorizer.pkl"
)

joblib.dump(
    char_vectorizer,
    CHAR_VECTORIZER_PATH,
)

assert CHAR_VECTORIZER_PATH.exists(), (
    "Character vectorizer was not serialized."
)


# ==============================================================================
# 11. SERIALIZE OBJECTIVE VECTOR CACHE
# ==============================================================================

OBJECTIVE_CHAR_MATRIX_PATH = (
    R1_CHAR_ROOT
    / "objective_char_tfidf.npz"
)

sparse.save_npz(
    OBJECTIVE_CHAR_MATRIX_PATH,
    objective_char_matrix,
)

OBJECTIVE_CHAR_META_PATH = (
    R1_CHAR_ROOT
    / "objective_char_index.parquet"
)

objective_char[
    [
        OBJECTIVE_UID_COL,
    ]
].to_parquet(
    OBJECTIVE_CHAR_META_PATH,
    index=False,
)

assert OBJECTIVE_CHAR_MATRIX_PATH.exists()
assert OBJECTIVE_CHAR_META_PATH.exists()


# ==============================================================================
# 12. SANITY SIMILARITY CHECK
# ==============================================================================

print("\n" + "=" * 80)
print("CHARACTER SIMILARITY SANITY CHECK")
print("=" * 80)

sanity_objective = (
    "compare fractions with different denominators"
)

sanity_turns = [
    "compare fractions with different denominators",
    "the student explains why one fraction is larger",
    "the tutor asks about classroom routines",
]

sanity_objective_matrix = (
    char_vectorizer.transform(
        [
            normalize_char_text(
                sanity_objective
            )
        ]
    )
)

sanity_turn_matrix = (
    char_vectorizer.transform(
        [
            normalize_char_text(text)
            for text in sanity_turns
        ]
    )
)

sanity_scores = (
    sanity_objective_matrix
    @ sanity_turn_matrix.T
).toarray().ravel()

for text, score in zip(
    sanity_turns,
    sanity_scores,
):
    print(
        f"{score:8.4f} | {text}"
    )

assert sanity_scores[0] >= sanity_scores[1], (
    "Character similarity sanity check failed."
)


# ==============================================================================
# 13. TARGET / EVALUATION ISOLATION
# ==============================================================================

print("\n" + "=" * 80)
print("TARGET ISOLATION AUDIT")
print("=" * 80)

# R0 objective catalogue itself is already target-free.
# Cell 4 must not add any target-derived field.

FORBIDDEN_CHAR_FIELDS = {
    "target",
    "label",
    "prediction",
    "positive_rate",
    "target_mean",
    "target_count",
    "positive_count",
    "negative_count",
    "fold",
    "response_count",
    "session_count",
}

observed_char_fields = set(
    objective_char.columns
)

prohibited_char_fields = sorted(
    observed_char_fields
    & FORBIDDEN_CHAR_FIELDS
)

print(
    "Prohibited fields:",
    prohibited_char_fields,
)

assert prohibited_char_fields == [], (
    "Target/evaluation-derived field entered "
    "character retrieval artifact."
)


# ==============================================================================
# 14. MANIFEST
# ==============================================================================

char_manifest = {
    "stage": "R1_SPARSE_RETRIEVAL",
    "component": "character_tfidf",

    "source_turn_rows": int(
        turn_pf_char.metadata.num_rows
    ),

    "objective_rows": int(
        len(objective_char)
    ),

    "source_objective_text_column": OBJECTIVE_TEXT_COL,

    "derived_text_column": "char_text",

    "analyzer": "char",

    "ngram_range": list(
        CHAR_NGRAM_RANGE
    ),

    "min_df": int(
        CHAR_MIN_DF
    ),

    "max_features": int(
        CHAR_MAX_FEATURES
    ),

    "sublinear_tf": bool(
        CHAR_SUBLINEAR_TF
    ),

    "normalization": "l2",

    "dtype": "float32",

    "vocabulary_size": int(
        vocabulary_size
    ),

    "target_used": False,

    "label_used": False,

    "full_turn_matrix_materialized": False,

    "session_local_retrieval_required": True,
}

with open(
    R1_CHAR_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        char_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )

assert R1_CHAR_MANIFEST_PATH.exists()


# ==============================================================================
# 15. READY FLAGS
# ==============================================================================

R1_CHAR_VECTOR_SPACE_READY = True
R1_CHAR_OBJECTIVE_CACHE_READY = True
R1_CHAR_TARGET_FREE = True
R1_CHAR_SERIALIZATION_READY = True

R1_CELL_4_READY = all([
    R1_CHAR_VECTOR_SPACE_READY,
    R1_CHAR_OBJECTIVE_CACHE_READY,
    R1_CHAR_TARGET_FREE,
    R1_CHAR_SERIALIZATION_READY,
])


# ==============================================================================
# 16. FINAL STATUS
# ==============================================================================

print("\n" + "=" * 80)
print("R1 CELL 4 STATUS")
print("=" * 80)

print(
    "Character vector space ready :",
    R1_CHAR_VECTOR_SPACE_READY,
)

print(
    "Objective cache ready        :",
    R1_CHAR_OBJECTIVE_CACHE_READY,
)

print(
    "Target-free                  :",
    R1_CHAR_TARGET_FREE,
)

print(
    "Serialization ready          :",
    R1_CHAR_SERIALIZATION_READY,
)

print(
    "R1_CELL_4_READY              :",
    R1_CELL_4_READY,
)

assert R1_CELL_4_READY is True

print("=" * 80)
print(
    "R1 CELL 4 — CHARACTER SIMILARITY: PASS"
)
print("=" * 80)


# Cleanup
del objective_char_matrix
del objective_char
del sanity_objective_matrix
del sanity_turn_matrix
del char_vectorizer

gc.collect()

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 4 — CHARACTER SIMILARITY

R1 Cell 2 dependency : PASS
Math artifact        : PASS
Target isolation     : PASS

CHARACTER ARTIFACT ROOT
Root: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\char_artifacts

CHARACTER NORMALIZATION SELF-TEST
Self-tests passed: 4/4

OBJECTIVE CHARACTER CORPUS
Objectives: 398
Empty normalized objectives: 0

CANONICAL TURN SOURCE
Turn source: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\turns.parquet
Turn rows: 6,139,854

FIT CHARACTER TF-IDF
n-gram range : (3, 5)
min_df       : 2
max_features : 500000

Fitting against canonical turn corpus...

Character vocabulary: 500,000

OBJECTIVE CHARACTER MATRIX
Shape: (398, 500000)
Non-zero entries: 42206

CHARACTER SIMILARITY SANITY CHECK
  1.0000 | compare fractions with different denominators
  0.1358 | the student explains why one fraction is larger
  0.0040 | the tutor asks about cl

57

In [9]:
# ==============================================================================
# R1 DEBUG — CELL 1 WORD TF-IDF ARTIFACT DISCOVERY
# ==============================================================================

import types
from pathlib import Path

print("=" * 80)
print("TRACE THE ACE — R1 WORD TF-IDF ARTIFACT DISCOVERY")
print("=" * 80)

print("\n" + "=" * 80)
print("R1-RELATED GLOBALS")
print("=" * 80)

r1_globals = {}

for name, value in sorted(globals().items()):

    if (
        name.startswith("R1")
        or "TFIDF" in name.upper()
        or "TF_IDF" in name.upper()
        or "WORD" in name.upper()
        or "HASH" in name.upper()
        or "VECTORIZ" in name.upper()
    ):

        if name.startswith("_"):
            continue

        value_type = type(value).__name__

        try:
            if hasattr(value, "shape"):
                extra = f" | shape={value.shape}"
            elif isinstance(value, (list, tuple, dict, set)):
                extra = f" | len={len(value)}"
            elif isinstance(value, Path):
                extra = f" | exists={value.exists()}"
            else:
                extra = ""
        except Exception:
            extra = ""

        r1_globals[name] = (
            value_type,
            extra,
        )

        print(
            f"{name:45s} "
            f"{value_type:25s} "
            f"{extra}"
        )


# ==============================================================================
# OBJECT TYPE INSPECTION
# ==============================================================================

print("\n" + "=" * 80)
print("POSSIBLE WORD TF-IDF OBJECTS")
print("=" * 80)

interesting_names = []

for name, (value_type, extra) in r1_globals.items():

    value = globals().get(name)

    if value is None:
        continue

    cls_name = type(value).__name__.lower()

    keywords = [
        "hashing",
        "tfidf",
        "transform",
        "vector",
        "sparse",
        "matrix",
        "transformer",
    ]

    if any(
        k in name.lower()
        for k in keywords
    ) or any(
        k in cls_name
        for k in keywords
    ):

        interesting_names.append(name)


for name in interesting_names:

    value = globals()[name]

    print("\n" + "-" * 80)
    print("NAME :", name)
    print("TYPE :", type(value))

    try:
        print(
            "SHAPE:",
            value.shape,
        )
    except Exception:
        pass

    # --------------------------------------------------------------
    # Public attributes only
    # --------------------------------------------------------------

    try:

        attrs = [
            a
            for a in dir(value)
            if not a.startswith("_")
        ]

        useful_attrs = [
            a
            for a in attrs
            if any(
                token in a.lower()
                for token in [
                    "transform",
                    "idf",
                    "vocab",
                    "dimension",
                    "feature",
                    "fold",
                    "fit",
                    "schema",
                    "path",
                ]
            )
        ]

        print(
            "USEFUL ATTRIBUTES:",
            useful_attrs[:80],
        )

    except Exception as exc:

        print(
            "Attribute inspection failed:",
            repr(exc),
        )


# ==============================================================================
# R1 OUTPUT TREE
# ==============================================================================

print("\n" + "=" * 80)
print("R1 OUTPUT TREE")
print("=" * 80)

try:

    r1_root_path = Path(R1_ROOT)

    print(
        "R1_ROOT:",
        r1_root_path,
    )

    if r1_root_path.exists():

        files = sorted(
            p
            for p in r1_root_path.rglob("*")
            if p.is_file()
        )

        for path in files[:300]:

            try:
                size_mb = (
                    path.stat().st_size
                    / (1024 ** 2)
                )
            except Exception:
                size_mb = None

            print(
                f"{path.relative_to(r1_root_path)}"
                f" | {size_mb:.3f} MB"
                if size_mb is not None
                else
                f"{path.relative_to(r1_root_path)}"
            )

    else:

        print(
            "R1_ROOT does not exist."
        )

except Exception as exc:

    print(
        "R1 tree inspection failed:",
        repr(exc),
    )


# ==============================================================================
# FOLD-SPECIFIC OBJECTS
# ==============================================================================

print("\n" + "=" * 80)
print("FOLD-SPECIFIC OBJECT DISCOVERY")
print("=" * 80)

for name, value in sorted(
    globals().items()
):

    lname = name.lower()

    if (
        "fold" in lname
        and (
            "tf" in lname
            or "idf" in lname
            or "word" in lname
            or "hash" in lname
            or "vector" in lname
        )
    ):

        print(
            f"{name:45s}"
            f" | type={type(value).__name__}"
        )

        try:
            print(
                "  shape =",
                value.shape,
            )
        except Exception:
            pass

        try:
            if isinstance(
                value,
                dict,
            ):
                print(
                    "  keys =",
                    list(value.keys())[:20],
                )
        except Exception:
            pass


print("\n" + "=" * 80)
print("R1 WORD TF-IDF DISCOVERY COMPLETE")
print("=" * 80)

TRACE THE ACE — R1 WORD TF-IDF ARTIFACT DISCOVERY

R1-RELATED GLOBALS
CHAR_VECTORIZER_PATH                          WindowsPath                | exists=True
HashingVectorizer                             type                      
NUMBER_WORDS                                  set                        | len=39
R1_CELL_0_READY                               bool                      
R1_CELL_1_READY                               bool                      
R1_CELL_2_READY                               bool                      
R1_CELL_4_READY                               bool                      
R1_CHAR_MANIFEST_PATH                         WindowsPath                | exists=True
R1_CHAR_OBJECTIVE_CACHE_READY                 bool                      
R1_CHAR_ROOT                                  WindowsPath                | exists=True
R1_CHAR_SERIALIZATION_READY                   bool                      
R1_CHAR_TARGET_FREE                           bool                      
R1_

In [12]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 5 — SESSION-LOCAL RETRIEVAL
# ==============================================================================

import gc
import json
import hashlib
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import sparse


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 5 — SESSION-LOCAL RETRIEVAL")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY GATES
# ==============================================================================

assert R1_CELL_0_READY is True, (
    "R1 Cell 0 must pass."
)

assert R1_CELL_1_READY is True, (
    "R1 Cell 1 TF-IDF build must pass."
)

assert R1_CELL_2_READY is True, (
    "R1 Cell 2 math extraction must pass."
)

assert R1_CELL_4_READY is True, (
    "R1 Cell 4 character similarity must pass."
)

# ==============================================================================
# R0 ARTIFACT-LEVEL DEPENDENCY VERIFICATION
#
# Do NOT depend on Python flags from the previous R0 notebook/kernel.
# Verify the frozen R0 artifacts directly from disk.
# ==============================================================================

R0_QUERIES_PATH = (
    R0_ROOT / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX_PATH = (
    R0_ROOT / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE_PATH = (
    R0_ROOT / "objective_catalogue.parquet"
)

R0_MANIFEST_PATH = (
    R0_ROOT / "r0_manifest.json"
)


# --------------------------------------------------------------------------
# Required frozen artifacts
# --------------------------------------------------------------------------

R0_REQUIRED_ARTIFACTS = {
    "retrieval_queries": R0_QUERIES_PATH,
    "session_turn_index": R0_SESSION_TURN_INDEX_PATH,
    "objective_catalogue": R0_OBJECTIVE_CATALOGUE_PATH,
    "r0_manifest": R0_MANIFEST_PATH,
}

R0_MISSING_ARTIFACTS = [
    name
    for name, path in R0_REQUIRED_ARTIFACTS.items()
    if not path.exists()
]

assert R0_MISSING_ARTIFACTS == [], (
    "Required frozen R0 artifacts are missing: "
    f"{R0_MISSING_ARTIFACTS}"
)


# --------------------------------------------------------------------------
# Manifest validity
# --------------------------------------------------------------------------

with open(
    R0_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    r0_manifest_check = json.load(f)


assert isinstance(
    r0_manifest_check,
    dict,
), "R0 manifest is not a JSON object."


# --------------------------------------------------------------------------
# Exact population checks
# --------------------------------------------------------------------------

r0_queries_rows = int(
    pq.ParquetFile(
        R0_QUERIES_PATH
    ).metadata.num_rows
)

r0_turn_rows = int(
    pq.ParquetFile(
        R0_SESSION_TURN_INDEX_PATH
    ).metadata.num_rows
)

r0_objective_rows = int(
    pq.ParquetFile(
        R0_OBJECTIVE_CATALOGUE_PATH
    ).metadata.num_rows
)


assert r0_queries_rows == 35_072, (
    f"R0 retrieval query population changed: "
    f"{r0_queries_rows}"
)

assert r0_turn_rows == 6_139_854, (
    f"R0 session-turn population changed: "
    f"{r0_turn_rows}"
)

assert r0_objective_rows == 398, (
    f"R0 objective population changed: "
    f"{r0_objective_rows}"
)


# --------------------------------------------------------------------------
# R0 artifact-level readiness
# --------------------------------------------------------------------------

R0_RETRIEVAL_INPUT_READY = True
R0_FROZEN = True


print("\n" + "=" * 80)
print("R0 ARTIFACT-LEVEL DEPENDENCY VERIFICATION")
print("=" * 80)

for name, path in R0_REQUIRED_ARTIFACTS.items():
    print(
        f"{name:24s}:",
        path.exists(),
    )

print(
    "\nRetrieval queries:",
    f"{r0_queries_rows:,}",
)

print(
    "Session-turn rows:",
    f"{r0_turn_rows:,}",
)

print(
    "Objectives:",
    f"{r0_objective_rows:,}",
)

print(
    "R0 manifest JSON:",
    "VALID",
)

print(
    "R0_RETRIEVAL_INPUT_READY:",
    R0_RETRIEVAL_INPUT_READY,
)

print(
    "R0_FROZEN:",
    R0_FROZEN,
)

assert R0_RETRIEVAL_INPUT_READY is True
assert R0_FROZEN is True

print("=" * 80)
print("R0 ARTIFACT DEPENDENCY: PASS")
print("=" * 80)


print("\nR1 Cell 0 dependency : PASS")
print("R1 Cell 1 dependency : PASS")
print("R1 Cell 2 dependency : PASS")
print("R1 Cell 4 dependency : PASS")
print("R0 retrieval input   : PASS")
print("R0 frozen            : PASS")


# ==============================================================================
# 1. R0 ARTIFACT PATHS
# ==============================================================================

R0_QUERIES_PATH = (
    R0_ROOT / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX_PATH = (
    R0_ROOT / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE_PATH = (
    R0_ROOT / "objective_catalogue.parquet"
)


for path in [
    R0_QUERIES_PATH,
    R0_SESSION_TURN_INDEX_PATH,
    R0_OBJECTIVE_CATALOGUE_PATH,
]:
    assert path.exists(), (
        f"Required R0 artifact missing: {path}"
    )


# ==============================================================================
# 2. R1 CHARACTER ARTIFACTS
# ==============================================================================

R1_CHAR_ROOT = (
    R1_ROOT / "char_artifacts"
)

CHAR_VECTORIZER_PATH = (
    R1_CHAR_ROOT
    / "character_tfidf_vectorizer.pkl"
)

OBJECTIVE_CHAR_MATRIX_PATH = (
    R1_CHAR_ROOT
    / "objective_char_tfidf.npz"
)

OBJECTIVE_CHAR_META_PATH = (
    R1_CHAR_ROOT
    / "objective_char_index.parquet"
)

assert CHAR_VECTORIZER_PATH.exists()
assert OBJECTIVE_CHAR_MATRIX_PATH.exists()
assert OBJECTIVE_CHAR_META_PATH.exists()


# ==============================================================================
# 3. R1 MATH ARTIFACTS
# ==============================================================================

R1_MATH_ROOT = (
    R1_ROOT / "math_artifacts"
)

R1_TURN_MATH_PATH = (
    R1_MATH_ROOT
    / "turn_math_tokens.parquet"
)

R1_OBJECTIVE_MATH_PATH = (
    R1_MATH_ROOT
    / "objective_math_tokens.parquet"
)

assert R1_TURN_MATH_PATH.exists()
assert R1_OBJECTIVE_MATH_PATH.exists()


# ==============================================================================
# 4. SESSION-LOCAL RETRIEVAL CONFIG
# ==============================================================================

# --------------------------------------------------------------------------
# IMPORTANT:
#
# Cell 5 deliberately does NOT select Top-K.
#
# It computes session-local objective ↔ turn scores.
# Cell 6 performs Top-K selection.
#
# To avoid a giant permanent objective × turn table, only one response's
# session is scored at a time and the scored block is passed to Cell 6.
# --------------------------------------------------------------------------

WORD_WEIGHT = 0.50
CHAR_WEIGHT = 0.30
MATH_WEIGHT = 0.20

assert np.isclose(
    WORD_WEIGHT
    + CHAR_WEIGHT
    + MATH_WEIGHT,
    1.0,
)


print("\n" + "=" * 80)
print("R1 RETRIEVAL CONFIGURATION")
print("=" * 80)

print(
    "Word TF-IDF weight :",
    WORD_WEIGHT,
)

print(
    "Character weight   :",
    CHAR_WEIGHT,
)

print(
    "Math overlap weight:",
    MATH_WEIGHT,
)

print(
    "Retrieval scope    : SAME SESSION ONLY"
)

print(
    "Top-K selection    : CELL 6"
)


# ==============================================================================
# 5. LOAD R0 QUERY TABLE
# ==============================================================================

retrieval_queries_r1 = pd.read_parquet(
    R0_QUERIES_PATH
)

assert len(
    retrieval_queries_r1
) == 35_072

assert retrieval_queries_r1[
    "response_id"
].is_unique

assert retrieval_queries_r1[
    "session_id"
].notna().all()

assert retrieval_queries_r1[
    "objective_uid"
].notna().all()

assert retrieval_queries_r1[
    "fold"
].isin([0, 1, 2, 3, 4]).all()


# ==============================================================================
# 6. LOAD R0 SESSION-TURN INDEX
# ==============================================================================

session_turn_index_r1 = pd.read_parquet(
    R0_SESSION_TURN_INDEX_PATH
)

assert len(
    session_turn_index_r1
) == 6_139_854

assert session_turn_index_r1[
    "turn_uid"
].is_unique

assert session_turn_index_r1[
    "session_id"
].notna().all()

assert session_turn_index_r1[
    "text_norm"
].notna().all()


# ==============================================================================
# 7. LOAD OBJECTIVE CATALOGUE
# ==============================================================================

objective_catalogue_r1 = pd.read_parquet(
    R0_OBJECTIVE_CATALOGUE_PATH
)

assert len(
    objective_catalogue_r1
) == 398

assert objective_catalogue_r1[
    "objective_uid"
].is_unique

assert objective_catalogue_r1[
    "objective_raw"
].notna().all()


# ==============================================================================
# 8. LOAD CHARACTER VECTOR SPACE
# ==============================================================================

print("\n" + "=" * 80)
print("LOAD CHARACTER RETRIEVAL SPACE")
print("=" * 80)

char_vectorizer_r1 = joblib.load(
    CHAR_VECTORIZER_PATH
)

objective_char_matrix_r1 = (
    sparse.load_npz(
        OBJECTIVE_CHAR_MATRIX_PATH
    )
)

objective_char_meta_r1 = pd.read_parquet(
    OBJECTIVE_CHAR_META_PATH
)

assert (
    objective_char_matrix_r1.shape[0]
    == len(objective_char_meta_r1)
)

assert (
    objective_char_meta_r1[
        "objective_uid"
    ].is_unique
)

print(
    "Character vocabulary:",
    f"{len(char_vectorizer_r1.vocabulary_):,}"
)

print(
    "Objective matrix:",
    objective_char_matrix_r1.shape
)


# ==============================================================================
# 9. LOAD MATH SIGNATURES
# ==============================================================================

objective_math_r1 = pd.read_parquet(
    R1_OBJECTIVE_MATH_PATH
)

assert len(
    objective_math_r1
) == 398

assert objective_math_r1[
    "objective_uid"
].is_unique


# Build objective_uid → token-set mapping.

objective_math_sets = {
    row.objective_uid: (
        set(
            str(row.math_tokens).split()
        )
        if row.math_tokens
        else set()
    )
    for row in objective_math_r1.itertuples(
        index=False
    )
}


# ==============================================================================
# 10. TURN MATH INDEX
#
# We create a lightweight session-local lookup:
#
#     session_id → turn_uid → math token set
#
# The actual text remains in session_turn_index.
# ==============================================================================

turn_math_r1 = pd.read_parquet(
    R1_TURN_MATH_PATH,
)

assert len(
    turn_math_r1
) == 6_139_854

assert turn_math_r1[
    "turn_uid"
].is_unique


turn_math_r1[
    "math_tokens"
] = turn_math_r1[
    "math_tokens"
].fillna("")


turn_math_r1[
    "math_token_set"
] = (
    turn_math_r1[
        "math_tokens"
    ]
    .map(
        lambda x: set(
            str(x).split()
        ) if str(x).strip()
        else set()
    )
)


# ==============================================================================
# 11. CHARACTER NORMALIZATION
# ==============================================================================

def normalize_char_text_r1(text):

    if text is None:
        return ""

    text = str(text).lower()

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


# ==============================================================================
# 12. OBJECTIVE CHARACTER TEXT
#
# R0 catalogue has objective_raw only.
# Character-normalized text is derived here.
# ==============================================================================

objective_catalogue_r1[
    "char_text"
] = (
    objective_catalogue_r1[
        "objective_raw"
    ]
    .map(
        normalize_char_text_r1
    )
)


# ==============================================================================
# 13. OBJECTIVE CHARACTER MATRIX ORDER CONTRACT
# ==============================================================================

objective_char_uid_to_row = {
    uid: i
    for i, uid in enumerate(
        objective_char_meta_r1[
            "objective_uid"
        ]
    )
}

assert set(
    objective_char_uid_to_row
) == set(
    objective_catalogue_r1[
        "objective_uid"
    ]
)


# ==============================================================================
# 14. RESPONSE → OBJECTIVE CONTRACT
# ==============================================================================

objective_raw_lookup = (
    objective_catalogue_r1
    .set_index(
        "objective_uid"
    )[
        "objective_raw"
    ]
    .to_dict()
)

raw_mismatches = (
    retrieval_queries_r1.apply(
        lambda row:
            row["objective_raw"]
            != objective_raw_lookup[
                row["objective_uid"]
            ],
        axis=1,
    )
    .sum()
)

assert raw_mismatches == 0, (
    "Response objective_raw does not match "
    "objective catalogue."
)


# ==============================================================================
# 15. FOLD CONTRACT
# ==============================================================================

session_fold_counts = (
    retrieval_queries_r1[
        [
            "session_id",
            "fold",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "session_id"
    )[
        "fold"
    ]
    .nunique()
)

assert (
    session_fold_counts.max()
    == 1
), (
    "A session maps to multiple folds."
)

assert (
    len(session_fold_counts)
    == 22_821
)


# ==============================================================================
# 16. SESSION INDEX
#
# Group only by session_id.
# This is the hard boundary preventing cross-session retrieval.
# ==============================================================================

session_turn_groups = {
    session_id: group
    for session_id, group
    in session_turn_index_r1.groupby(
        "session_id",
        sort=False,
    )
}


# ==============================================================================
# 17. RETRIEVAL SCORING FUNCTION
# ==============================================================================

def math_overlap_score(
    objective_tokens,
    turn_tokens,
):
    """
    Jaccard overlap over extracted mathematical signatures.

    If neither side contains math tokens, return 0.
    """

    if not objective_tokens:
        return 0.0

    if not turn_tokens:
        return 0.0

    union = (
        objective_tokens
        | turn_tokens
    )

    if not union:
        return 0.0

    return float(
        len(
            objective_tokens
            & turn_tokens
        )
        / len(union)
    )


def score_one_response_session(
    response_row,
    max_turns_per_session=None,
):
    """
    Score exactly one response against turns from its own session.

    Returns a DataFrame containing:
        response_id
        session_id
        objective_uid
        fold
        turn_uid
        role
        turn_index
        word_score
        char_score
        math_score
        sparse_score
    """

    response_id = (
        response_row["response_id"]
    )

    session_id = (
        response_row["session_id"]
    )

    objective_uid = (
        response_row["objective_uid"]
    )

    fold = int(
        response_row["fold"]
    )

    # --------------------------------------------------------------
    # HARD SESSION BOUNDARY
    # --------------------------------------------------------------

    turns_local = session_turn_groups.get(
        session_id
    )

    if turns_local is None:
        return pd.DataFrame()

    if max_turns_per_session is not None:
        turns_local = turns_local.head(
            max_turns_per_session
        )

    # --------------------------------------------------------------
    # Objective row / character vector
    # --------------------------------------------------------------

    objective_row_index = (
        objective_char_uid_to_row[
            objective_uid
        ]
    )

    objective_char_vector = (
        objective_char_matrix_r1[
            objective_row_index
        ]
    )

    objective_math_tokens = (
        objective_math_sets.get(
            objective_uid,
            set(),
        )
    )

    # --------------------------------------------------------------
    # Turn character vectors
    # --------------------------------------------------------------

    turn_texts = (
        turns_local[
            "text_norm"
        ]
        .map(
            normalize_char_text_r1
        )
        .tolist()
    )

    turn_char_matrix = (
        char_vectorizer_r1.transform(
            turn_texts
        )
    )

    char_scores = (
        objective_char_vector
        @ turn_char_matrix.T
    ).toarray().ravel()

    # --------------------------------------------------------------
    # Math scores
    # --------------------------------------------------------------

    turn_math_lookup = (
        turn_math_r1[
            turn_math_r1[
                "turn_uid"
            ].isin(
                turns_local[
                    "turn_uid"
                ]
            )
        ][
            [
                "turn_uid",
                "math_token_set",
            ]
        ]
        .set_index(
            "turn_uid"
        )[
            "math_token_set"
        ]
        .to_dict()
    )

    math_scores = np.asarray(
        [
            math_overlap_score(
                objective_math_tokens,
                turn_math_lookup.get(
                    turn_uid,
                    set(),
                ),
            )
            for turn_uid
            in turns_local[
                "turn_uid"
            ]
        ],
        dtype=np.float32,
    )

    # --------------------------------------------------------------
    # Word score placeholder
    #
    # Cell 1's fold-aware word TF-IDF implementation is intentionally
    # not silently replaced here.
    #
    # If Cell 1 exposed a compatible scoring function, use it.
    # Otherwise fail explicitly rather than pretending char+math is
    # the complete sparse architecture.
    # --------------------------------------------------------------

    if "r1_word_score_session" in globals():

        word_scores = np.asarray(
            r1_word_score_session(
                response_row=response_row,
                turns_local=turns_local,
            ),
            dtype=np.float32,
        )

    else:

        raise RuntimeError(
            "Cell 1 did not expose "
            "'r1_word_score_session'. "
            "Do not continue with a char+math-only "
            "approximation. Expose the fold-aware "
            "word TF-IDF scorer from Cell 1 first."
        )

    # --------------------------------------------------------------
    # Final sparse score
    # --------------------------------------------------------------

    sparse_scores = (
        WORD_WEIGHT * word_scores
        + CHAR_WEIGHT * char_scores.astype(
            np.float32
        )
        + MATH_WEIGHT * math_scores
    )

    result = pd.DataFrame(
        {
            "response_id": response_id,
            "session_id": session_id,
            "objective_uid": objective_uid,
            "fold": fold,
            "turn_uid": (
                turns_local[
                    "turn_uid"
                ].to_numpy()
            ),
            "role": (
                turns_local[
                    "role"
                ].to_numpy()
            ),
            "turn_index": (
                turns_local[
                    "turn_index"
                ].to_numpy()
            ),
            "word_score": word_scores,
            "char_score": (
                char_scores.astype(
                    np.float32
                )
            ),
            "math_score": math_scores,
            "sparse_score": sparse_scores,
        }
    )

    return result


# ==============================================================================
# 18. DO NOT EXECUTE FULL CORPUS YET
#
# Cell 5 establishes and smoke-tests the session-local scoring engine.
# Cell 6 will perform controlled Top-K generation.
#
# This prevents a 35,072-response × 6.1M-turn accidental workload.
# ==============================================================================

print("\n" + "=" * 80)
print("SESSION-LOCAL RETRIEVAL ENGINE READY")
print("=" * 80)

print(
    "Response rows available:",
    f"{len(retrieval_queries_r1):,}",
)

print(
    "Session groups:",
    f"{len(session_turn_groups):,}",
)

print(
    "Canonical turns:",
    f"{len(session_turn_index_r1):,}",
)

print(
    "Objective catalogue:",
    f"{len(objective_catalogue_r1):,}",
)

print(
    "\nGlobal Cartesian retrieval: FORBIDDEN"
)

print(
    "Session-local boundary: ENFORCED"
)


# ==============================================================================
# 19. SMOKE TEST — FIRST RESPONSE ONLY
# ==============================================================================

print("\n" + "=" * 80)
print("SESSION-LOCAL RETRIEVAL SMOKE TEST")
print("=" * 80)


smoke_response = (
    retrieval_queries_r1
    .sort_values(
        "response_id"
    )
    .iloc[0]
)

smoke_session = (
    smoke_response[
        "session_id"
    ]
)

smoke_turns = session_turn_groups[
    smoke_session
]

print(
    "response_id:",
    smoke_response[
        "response_id"
    ],
)

print(
    "session_id:",
    smoke_session,
)

print(
    "objective_uid:",
    smoke_response[
        "objective_uid"
    ],
)

print(
    "Session-local turns:",
    f"{len(smoke_turns):,}",
)


smoke_result = score_one_response_session(
    smoke_response
)


assert len(
    smoke_result
) == len(
    smoke_turns
), (
    "Smoke-test score population does not equal "
    "the response session turn population."
)

assert (
    smoke_result[
        "session_id"
    ].nunique()
    == 1
)

assert (
    smoke_result[
        "session_id"
    ].iloc[0]
    == smoke_session
)

assert (
    smoke_result[
        "turn_uid"
    ].is_unique
)

assert (
    smoke_result[
        "sparse_score"
    ].notna().all()
)

assert (
    smoke_result[
        "word_score"
    ].between(
        0,
        1,
    ).all()
)

assert (
    smoke_result[
        "char_score"
    ].between(
        0,
        1,
    ).all()
)

assert (
    smoke_result[
        "math_score"
    ].between(
        0,
        1,
    ).all()
)


print(
    "\nSmoke scored rows:",
    f"{len(smoke_result):,}",
)

print(
    "Unique sessions:",
    smoke_result[
        "session_id"
    ].nunique(),
)

print(
    "Session boundary: PASS"
)

print(
    "Score validity: PASS"
)

print(
    "\nTop smoke scores:"
)

display(
    smoke_result
    .sort_values(
        "sparse_score",
        ascending=False,
    )
    .head(10)
)


# ==============================================================================
# 20. CROSS-SESSION CONTAMINATION TEST
# ==============================================================================

print("\n" + "=" * 80)
print("CROSS-SESSION CONTAMINATION TEST")
print("=" * 80)

assert not (
    set(
        smoke_result[
            "session_id"
        ]
    )
    - {
        smoke_session
    }
), (
    "Cross-session turn entered retrieval result."
)

print(
    "Foreign-session turns:",
    0,
)

print(
    "Cross-session contamination: PASS"
)


# ==============================================================================
# 21. TARGET ISOLATION
# ==============================================================================

result_columns = set(
    smoke_result.columns
)

PROHIBITED_R1_RESULT_FIELDS = {
    "target",
    "label",
    "is_correct",
    "positive_rate",
    "target_mean",
    "target_count",
    "positive_count",
    "negative_count",
}

assert not (
    result_columns
    & PROHIBITED_R1_RESULT_FIELDS
), (
    "Target/evaluation field entered sparse retrieval result."
)


# ==============================================================================
# 22. CELL 5 READY
# ==============================================================================

R1_SESSION_LOCAL_RETRIEVAL_READY = True
R1_SESSION_BOUNDARY_READY = True
R1_SCORING_CONTRACT_READY = True
R1_CELL_5_READY = True


print("\n" + "=" * 80)
print("R1 CELL 5 STATUS")
print("=" * 80)

print(
    "Session-local retrieval ready :",
    R1_SESSION_LOCAL_RETRIEVAL_READY,
)

print(
    "Session boundary ready        :",
    R1_SESSION_BOUNDARY_READY,
)

print(
    "Scoring contract ready        :",
    R1_SCORING_CONTRACT_READY,
)

print(
    "R1_CELL_5_READY               :",
    R1_CELL_5_READY,
)

assert R1_CELL_5_READY is True

print("=" * 80)
print("R1 CELL 5 — SESSION-LOCAL RETRIEVAL: PASS")
print("=" * 80)


# Cleanup only large objects that are not needed by the scoring function.
del smoke_result
gc.collect()

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 5 — SESSION-LOCAL RETRIEVAL

R0 ARTIFACT-LEVEL DEPENDENCY VERIFICATION
retrieval_queries       : True
session_turn_index      : True
objective_catalogue     : True
r0_manifest             : True

Retrieval queries: 35,072
Session-turn rows: 6,139,854
Objectives: 398
R0 manifest JSON: VALID
R0_RETRIEVAL_INPUT_READY: True
R0_FROZEN: True
R0 ARTIFACT DEPENDENCY: PASS

R1 Cell 0 dependency : PASS
R1 Cell 1 dependency : PASS
R1 Cell 2 dependency : PASS
R1 Cell 4 dependency : PASS
R0 retrieval input   : PASS
R0 frozen            : PASS

R1 RETRIEVAL CONFIGURATION
Word TF-IDF weight : 0.5
Character weight   : 0.3
Math overlap weight: 0.2
Retrieval scope    : SAME SESSION ONLY
Top-K selection    : CELL 6

LOAD CHARACTER RETRIEVAL SPACE
Character vocabulary: 500,000
Objective matrix: (398, 500000)

SESSION-LOCAL RETRIEVAL ENGINE READY
Response rows available: 35,072
Session groups: 22,821
Canonical turns: 6,139,854
Objective catalogue: 398

Global Cartes

,response_id,session_id,objective_uid,fold,turn_uid,role,turn_index,word_score,char_score,math_score,sparse_score
54,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,34f3d7a9b6d849047da85341e286e6ae5c15e9386f9fe0...,student,54,0.934460,0.938481,1.0,0.948774
58,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,6509058c57bd70f5890d0c5f0e689ffae1706db65560af...,student,58,0.296735,0.277684,1.0,0.431673
132,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,1dcfd22d25e4b1e4b61a43078cfe2fad5b7186e56cb009...,tutor,132,0.012483,0.030394,1.0,0.215360
64,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,0696e3412ccf6d960e42a89d5f5216e52ff3b74ac3c81e...,student,64,0.235356,0.240355,0.0,0.189784
93,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,d1e5a5f2b3aaf0f1835b5e3e1bd6cb9ea68aef0ab45390...,student,93,0.176896,0.326919,0.0,0.186524
56,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,13f3ae59594668064553ab9ae39d4dd0b3987821bd13a9...,student,56,0.157521,0.295715,0.0,0.167475
94,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,3aec17e2777c8090f4a6b29c79fef6c11551bbe51b39dd...,tutor,94,0.116063,0.207862,0.0,0.120390
246,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,262f39d864ffb907aee4af8197e9ca3d6b3eeabff81b38...,student,246,0.130018,0.183878,0.0,0.120173
241,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,8ec65f12632390a2ed3b4e39268f2350fb436b06a3bcb5...,tutor,241,0.138712,0.168551,0.0,0.119921
302,aaaavsh,bcaufvc,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3,8c5b6c207b21be9fc14bdd6d184034d5e17b3776f5aca5...,tutor,302,0.137691,0.164210,0.0,0.118109



CROSS-SESSION CONTAMINATION TEST
Foreign-session turns: 0
Cross-session contamination: PASS

R1 CELL 5 STATUS
Session-local retrieval ready : True
Session boundary ready        : True
Scoring contract ready        : True
R1_CELL_5_READY               : True
R1 CELL 5 — SESSION-LOCAL RETRIEVAL: PASS


0

In [15]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 6 — TOP-K CANDIDATE SELECTION
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 6 — TOP-K CANDIDATE SELECTION")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY GATES
# ==============================================================================

assert R1_CELL_0_READY is True, (
    "R1 Cell 0 must pass."
)

assert R1_CELL_1_READY is True, (
    "R1 Cell 1 must pass."
)

assert R1_CELL_1B_READY is True, (
    "R1 Cell 1B word scorer must pass."
)

assert R1_CELL_2_READY is True, (
    "R1 Cell 2 math extraction must pass."
)

assert R1_CELL_4_READY is True, (
    "R1 Cell 4 character similarity must pass."
)

assert R1_CELL_5_READY is True, (
    "R1 Cell 5 session-local retrieval must pass."
)

assert R0_RETRIEVAL_INPUT_READY is True, (
    "R0 retrieval input is not ready."
)

assert R0_FROZEN is True, (
    "R0 is not frozen."
)


print("\nR1 Cell 0 dependency : PASS")
print("R1 Cell 1B dependency: PASS")
print("R1 Cell 2 dependency : PASS")
print("R1 Cell 4 dependency : PASS")
print("R1 Cell 5 dependency : PASS")


# ==============================================================================
# 1. TOP-K CONFIGURATION
# ==============================================================================
#
# Sparse retrieval is only a candidate generator.
#
# We deliberately keep K moderate so downstream reranking can inspect
# a small but useful candidate set.
#
# ==============================================================================

R1_TOP_K = 50

assert (
    isinstance(
        R1_TOP_K,
        int,
    )
    and R1_TOP_K > 0
)

print("\n" + "=" * 80)
print("TOP-K CONFIGURATION")
print("=" * 80)

print(
    "Top-K per response:",
    R1_TOP_K,
)


# ==============================================================================
# 2. REQUIRED INPUT CONTRACT
# ==============================================================================

assert (
    "retrieval_queries_r1" in globals()
), (
    "retrieval_queries_r1 is not available."
)

assert (
    "session_turn_groups" in globals()
), (
    "session_turn_groups is not available."
)

assert callable(
    score_one_response_session
), (
    "score_one_response_session is not available."
)


required_query_columns = {
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
}

missing_query_columns = (
    required_query_columns
    -
    set(
        retrieval_queries_r1.columns
    )
)

assert missing_query_columns == set(), (
    f"Missing retrieval query columns: "
    f"{sorted(missing_query_columns)}"
)


print(
    "\nInput responses:",
    f"{len(retrieval_queries_r1):,}",
)

print(
    "Input sessions:",
    f"{retrieval_queries_r1['session_id'].nunique():,}",
)


# ==============================================================================
# 3. TARGET / LABEL ISOLATION
# ==============================================================================

PROHIBITED_R1_COLUMNS = {
    "target",
    "label",
    "y",
    "score_target",
    "is_correct",
    "correct",
    "outcome",
}

observed_prohibited = (
    PROHIBITED_R1_COLUMNS
    &
    set(
        retrieval_queries_r1.columns
    )
)

assert observed_prohibited == set(), (
    "Target/evaluation columns leaked into "
    f"retrieval query input: {sorted(observed_prohibited)}"
)


# ==============================================================================
# 4. SINGLE-RESPONSE TOP-K FUNCTION
# ==============================================================================

def select_top_k_candidates(
    response_row,
    scored_session,
    top_k=R1_TOP_K,
):
    """
    Select deterministic Top-K candidates from one session.

    Expected scored_session columns:
        response_id
        session_id
        objective_uid
        fold
        turn_uid
        role
        turn_index
        word_score
        char_score
        math_score
        sparse_score

    Ranking:
        1. sparse_score descending
        2. turn_index ascending
        3. turn_uid ascending

    No target or label is used.
    """

    assert (
        "response_id"
        in scored_session.columns
    )

    assert (
        "session_id"
        in scored_session.columns
    )

    assert (
        "turn_uid"
        in scored_session.columns
    )

    assert (
        "turn_index"
        in scored_session.columns
    )

    assert (
        "sparse_score"
        in scored_session.columns
    )

    # --------------------------------------------------------------
    # Session isolation
    # --------------------------------------------------------------

    response_session_id = (
        response_row[
            "session_id"
        ]
    )

    foreign_rows = (
        scored_session[
            scored_session[
                "session_id"
            ]
            != response_session_id
        ]
    )

    assert len(foreign_rows) == 0, (
        "Cross-session rows entered Top-K selection."
    )

    # --------------------------------------------------------------
    # Score validity
    # --------------------------------------------------------------

    assert np.isfinite(
        scored_session[
            "sparse_score"
        ].to_numpy(
            dtype=np.float64
        )
    ).all(), (
        "Non-finite sparse score encountered."
    )

    # --------------------------------------------------------------
    # Deterministic ranking
    # --------------------------------------------------------------

    ranked = (
        scored_session
        .sort_values(
            by=[
                "sparse_score",
                "turn_index",
                "turn_uid",
            ],
            ascending=[
                False,
                True,
                True,
            ],
            kind="mergesort",
        )
        .head(
            int(top_k)
        )
        .copy()
    )

    return ranked


# ==============================================================================
# 5. SMOKE TEST
# ==============================================================================

print("\n" + "=" * 80)
print("TOP-K SMOKE TEST")
print("=" * 80)

smoke_response = (
    retrieval_queries_r1
    .iloc[0]
)

smoke_scored = (
    score_one_response_session(
        smoke_response
    )
)

smoke_topk = (
    select_top_k_candidates(
        response_row=smoke_response,
        scored_session=smoke_scored,
        top_k=R1_TOP_K,
    )
)

assert len(
    smoke_topk
) <= R1_TOP_K

assert (
    smoke_topk[
        "session_id"
    ]
    .nunique()
    == 1
)

assert (
    smoke_topk[
        "response_id"
    ]
    .nunique()
    == 1
)

assert (
    smoke_topk[
        "turn_uid"
    ].nunique()
    ==
    len(smoke_topk)
)

assert (
    smoke_topk[
        "sparse_score"
    ].is_monotonic_decreasing
), (
    "Top-K scores are not sorted deterministically."
)

print(
    "Response:",
    smoke_response[
        "response_id"
    ],
)

print(
    "Session:",
    smoke_response[
        "session_id"
    ],
)

print(
    "Full session scored:",
    len(smoke_scored),
)

print(
    "Top-K selected:",
    len(smoke_topk),
)

print(
    "Top score:",
    float(
        smoke_topk[
            "sparse_score"
        ].iloc[0]
    ),
)

print(
    "Bottom Top-K score:",
    float(
        smoke_topk[
            "sparse_score"
        ].iloc[-1]
    ),
)


# ==============================================================================
# 6. BUILD ALL TOP-K CANDIDATES
# ==============================================================================
#
# Important:
# We keep this sequential.
#
# The corpus has ~6.14M turns, but scoring is performed only inside each
# response's own session.
#
# ==============================================================================

print("\n" + "=" * 80)
print("BUILDING TOP-K CANDIDATES")
print("=" * 80)

candidate_chunks = []

total_responses = len(
    retrieval_queries_r1
)

for i, (_, response_row) in enumerate(
    retrieval_queries_r1.iterrows(),
    start=1,
):

    scored_session = (
        score_one_response_session(
            response_row
        )
    )

    topk = (
        select_top_k_candidates(
            response_row=response_row,
            scored_session=scored_session,
            top_k=R1_TOP_K,
        )
    )

    # --------------------------------------------------------------
    # Keep only candidate-generation fields.
    # --------------------------------------------------------------

    candidate = (
        topk[
            [
                "response_id",
                "session_id",
                "objective_uid",
                "fold",
                "turn_uid",
                "role",
                "turn_index",
                "word_score",
                "char_score",
                "math_score",
                "sparse_score",
            ]
        ]
        .copy()
    )

    candidate_chunks.append(
        candidate
    )

    if (
        i % 500 == 0
        or i == total_responses
    ):

        current_candidates = sum(
            len(x)
            for x in candidate_chunks
        )

        print(
            f"Processed {i:,}/{total_responses:,}"
            f" | candidates={current_candidates:,}"
        )


# ==============================================================================
# 7. CONCATENATE
# ==============================================================================

r1_sparse_candidates = pd.concat(
    candidate_chunks,
    ignore_index=True,
)

del candidate_chunks

gc.collect()


# ==============================================================================
# 8. POPULATION AUDIT
# ==============================================================================

expected_max_candidates = (
    total_responses
    * R1_TOP_K
)

assert (
    len(r1_sparse_candidates)
    <= expected_max_candidates
)

assert (
    len(r1_sparse_candidates)
    > 0
)

assert (
    r1_sparse_candidates[
        "response_id"
    ].nunique()
    == total_responses
), (
    "Some responses produced no candidate."
)


# Every response must have at most K.
candidate_counts = (
    r1_sparse_candidates
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
)

assert (
    candidate_counts.max()
    <= R1_TOP_K
)

assert (
    candidate_counts.min()
    > 0
)


# ==============================================================================
# 9. IDENTITY / RELATIONAL AUDIT
# ==============================================================================

assert (
    r1_sparse_candidates[
        "turn_uid"
    ].is_unique
    is False
), (
    "Global turn_uid uniqueness cannot be expected because "
    "the same turn may be a candidate for multiple responses."
)

assert (
    r1_sparse_candidates[
        [
            "response_id",
            "turn_uid",
        ]
    ]
    .duplicated()
    .sum()
    == 0
), (
    "Same response/turn pair appears more than once."
)


# Candidate session must equal response session.
response_session_lookup = (
    retrieval_queries_r1[
        [
            "response_id",
            "session_id",
        ]
    ]
    .drop_duplicates(
        "response_id"
    )
    .set_index(
        "response_id"
    )[
        "session_id"
    ]
)

candidate_expected_sessions = (
    r1_sparse_candidates[
        "response_id"
    ].map(
        response_session_lookup
    )
)

assert (
    candidate_expected_sessions
    ==
    r1_sparse_candidates[
        "session_id"
    ]
).all(), (
    "Candidate contains a session different from "
    "the response session."
)


# ==============================================================================
# 10. SCORE VALIDITY
# ==============================================================================

score_columns = [
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]

for col in score_columns:

    values = (
        r1_sparse_candidates[
            col
        ].to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        values
    ).all(), (
        f"Non-finite values in {col}."
    )


assert (
    r1_sparse_candidates[
        "word_score"
    ].between(
        -1.000001,
        1.000001,
    ).all()
)

assert (
    r1_sparse_candidates[
        "char_score"
    ].between(
        -1.000001,
        1.000001,
    ).all()
)

assert (
    r1_sparse_candidates[
        "math_score"
    ].between(
        -1.000001,
        1.000001,
    ).all()
)


# ==============================================================================
# 11. DETERMINISTIC ORDERING AUDIT
# ==============================================================================

ordering_failures = 0

for _, group in r1_sparse_candidates.groupby(
    "response_id",
    sort=False,
):

    expected = (
        group
        .sort_values(
            by=[
                "sparse_score",
                "turn_index",
                "turn_uid",
            ],
            ascending=[
                False,
                True,
                True,
            ],
            kind="mergesort",
        )
        .index
        .tolist()
    )

    actual = (
        group.index.tolist()
    )

    if actual != expected:
        ordering_failures += 1


assert (
    ordering_failures == 0
), (
    f"Top-K ordering failures: "
    f"{ordering_failures}"
)


# ==============================================================================
# 12. OUTPUT SUMMARY
# ==============================================================================

candidate_count_distribution = (
    candidate_counts
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 80)
print("R1 TOP-K CANDIDATE AUDIT")
print("=" * 80)

print(
    "Responses:",
    f"{total_responses:,}",
)

print(
    "Candidate rows:",
    f"{len(r1_sparse_candidates):,}",
)

print(
    "Maximum candidates/response:",
    int(
        candidate_counts.max()
    ),
)

print(
    "Minimum candidates/response:",
    int(
        candidate_counts.min()
    ),
)

print(
    "Unique response IDs:",
    r1_sparse_candidates[
        "response_id"
    ].nunique(),
)

print(
    "Unique candidate turns:",
    r1_sparse_candidates[
        "turn_uid"
    ].nunique(),
)

print(
    "Ordering failures:",
    ordering_failures,
)

print(
    "Cross-session contamination:",
    "0",
)


# ==============================================================================
# 13. FINAL CELL STATUS
# ==============================================================================

R1_TOPK_READY = True

print("\n" + "=" * 80)
print("R1 CELL 6 STATUS")
print("=" * 80)

print(
    "Top-K candidate selection ready :",
    R1_TOPK_READY,
)

assert R1_TOPK_READY is True

print(
    "=" * 80
)
print(
    "R1 CELL 6 — TOP-K CANDIDATE SELECTION: PASS"
)
print(
    "=" * 80
)

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 6 — TOP-K CANDIDATE SELECTION

R1 Cell 0 dependency : PASS
R1 Cell 1B dependency: PASS
R1 Cell 2 dependency : PASS
R1 Cell 4 dependency : PASS
R1 Cell 5 dependency : PASS

TOP-K CONFIGURATION
Top-K per response: 50

Input responses: 35,072
Input sessions: 22,821

TOP-K SMOKE TEST
Response: aaaavsh
Session: bcaufvc
Full session scored: 330
Top-K selected: 50
Top score: 0.9487743973731995
Bottom Top-K score: 0.023791037499904633

BUILDING TOP-K CANDIDATES
Processed 500/35,072 | candidates=24,964
Processed 1,000/35,072 | candidates=49,908
Processed 1,500/35,072 | candidates=74,908
Processed 2,000/35,072 | candidates=99,867
Processed 2,500/35,072 | candidates=124,825
Processed 3,000/35,072 | candidates=149,807
Processed 3,500/35,072 | candidates=174,773
Processed 4,000/35,072 | candidates=199,746
Processed 4,500/35,072 | candidates=224,720
Processed 5,000/35,072 | candidates=249,720
Processed 5,500/35,072 | candidates=274,687
Processed 6,000/35,072 

In [16]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 6F — FREEZE TOP-K CANDIDATE ARTIFACT
#
# IMPORTANT:
# This cell persists the completed Cell 6 computation.
# After this passes, Cell 6 must NOT be rerun.
# ==============================================================================

from pathlib import Path
import hashlib
import json
import os
import platform
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 6F — FREEZE TOP-K CANDIDATE ARTIFACT")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY
# ==============================================================================

assert "R1_TOPK_READY" in globals(), (
    "R1 Cell 6 has not completed."
)

assert R1_TOPK_READY is True, (
    "R1 Cell 6 is not ready."
)

assert "r1_sparse_candidates" in globals(), (
    "r1_sparse_candidates is not available in memory."
)


# ==============================================================================
# 1. OUTPUT ROOT
# ==============================================================================

R1_FREEZE_ROOT = (
    Path(SCRATCH_ROOT)
    / "02_retrieval"
    / "R1_sparse"
    / "frozen"
)

R1_FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R1_CANDIDATES_FINAL = (
    R1_FREEZE_ROOT
    / "r1_sparse_candidates.parquet"
)

R1_CANDIDATES_TMP = (
    R1_FREEZE_ROOT
    / ".r1_sparse_candidates.parquet.tmp"
)

R1_FREEZE_MANIFEST = (
    R1_FREEZE_ROOT
    / "r1_cell6_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("R1 FREEZE PATHS")
print("=" * 80)

print(
    "Freeze root:",
    R1_FREEZE_ROOT,
)

print(
    "Candidates:",
    R1_CANDIDATES_FINAL,
)

print(
    "Manifest:",
    R1_FREEZE_MANIFEST,
)


# ==============================================================================
# 2. EXACT COLUMN CONTRACT
# ==============================================================================

R1_CANDIDATE_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]

missing_columns = (
    set(R1_CANDIDATE_COLUMNS)
    -
    set(r1_sparse_candidates.columns)
)

assert missing_columns == set(), (
    f"Missing candidate columns: "
    f"{sorted(missing_columns)}"
)

unexpected_columns = (
    set(r1_sparse_candidates.columns)
    -
    set(R1_CANDIDATE_COLUMNS)
)

assert unexpected_columns == set(), (
    f"Unexpected candidate columns: "
    f"{sorted(unexpected_columns)}"
)

r1_sparse_candidates = (
    r1_sparse_candidates[
        R1_CANDIDATE_COLUMNS
    ]
    .copy()
)


# ==============================================================================
# 3. TARGET / LABEL ISOLATION
# ==============================================================================

PROHIBITED_COLUMNS = {
    "target",
    "label",
    "y",
    "correct",
    "is_correct",
    "outcome",
    "score_target",
}

observed_prohibited = (
    PROHIBITED_COLUMNS
    &
    set(r1_sparse_candidates.columns)
)

assert observed_prohibited == set(), (
    "Target/evaluation leakage detected in "
    f"candidate artifact: {sorted(observed_prohibited)}"
)


# ==============================================================================
# 4. POPULATION CONTRACT
# ==============================================================================

candidate_rows = len(
    r1_sparse_candidates
)

candidate_responses = (
    r1_sparse_candidates[
        "response_id"
    ].nunique()
)

candidate_sessions = (
    r1_sparse_candidates[
        "session_id"
    ].nunique()
)

candidate_turns = (
    r1_sparse_candidates[
        "turn_uid"
    ].nunique()
)

response_count = len(
    retrieval_queries_r1
)

assert candidate_responses == response_count, (
    "Not every retrieval response has candidates."
)

assert candidate_rows > 0

print("\n" + "=" * 80)
print("CANDIDATE POPULATION")
print("=" * 80)

print(
    "Candidate rows:",
    f"{candidate_rows:,}",
)

print(
    "Responses covered:",
    f"{candidate_responses:,}",
    "/",
    f"{response_count:,}",
)

print(
    "Candidate sessions:",
    f"{candidate_sessions:,}",
)

print(
    "Candidate turns:",
    f"{candidate_turns:,}",
)


# ==============================================================================
# 5. RELATIONAL CONTRACT
# ==============================================================================

response_session_lookup = (
    retrieval_queries_r1[
        [
            "response_id",
            "session_id",
        ]
    ]
    .drop_duplicates(
        "response_id"
    )
    .set_index(
        "response_id"
    )[
        "session_id"
    ]
)

candidate_expected_sessions = (
    r1_sparse_candidates[
        "response_id"
    ].map(
        response_session_lookup
    )
)

assert (
    candidate_expected_sessions
    ==
    r1_sparse_candidates[
        "session_id"
    ]
).all(), (
    "Candidate response/session relationship invalid."
)


# Same response + turn must never duplicate.
duplicate_pairs = (
    r1_sparse_candidates[
        [
            "response_id",
            "turn_uid",
        ]
    ]
    .duplicated()
    .sum()
)

assert duplicate_pairs == 0, (
    f"Duplicate response/turn pairs: {duplicate_pairs}"
)


# ==============================================================================
# 6. SCORE CONTRACT
# ==============================================================================

for column in [
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]:

    values = (
        r1_sparse_candidates[
            column
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        values
    ).all(), (
        f"Non-finite values in {column}."
    )


# ==============================================================================
# 7. FOLD CONTRACT
# ==============================================================================

observed_folds = sorted(
    r1_sparse_candidates[
        "fold"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

assert observed_folds == [0, 1, 2, 3, 4], (
    f"Unexpected folds: {observed_folds}"
)


# ==============================================================================
# 8. EXPLICIT ARROW SCHEMA
# ==============================================================================

candidate_table = pa.Table.from_pandas(
    r1_sparse_candidates,
    preserve_index=False,
)

print("\n" + "=" * 80)
print("SERIALIZATION SCHEMA")
print("=" * 80)

print(
    candidate_table.schema
)


# ==============================================================================
# 9. WRITE TEMPORARY PARQUET
# ==============================================================================

if R1_CANDIDATES_TMP.exists():
    try:
        R1_CANDIDATES_TMP.unlink()
        print(
            "\nRemoved stale temporary candidate artifact."
        )
    except PermissionError as exc:
        raise RuntimeError(
            "Temporary candidate parquet is locked. "
            "Restart the kernel before retrying."
        ) from exc


pq.write_table(
    candidate_table,
    R1_CANDIDATES_TMP,
    compression="zstd",
    use_dictionary=True,
)


assert R1_CANDIDATES_TMP.exists(), (
    "Temporary candidate parquet was not created."
)


# ==============================================================================
# 10. ROUND-TRIP VERIFICATION
# ==============================================================================

reload_table = pq.read_table(
    R1_CANDIDATES_TMP
)

reload_df = (
    reload_table
    .to_pandas()
)

assert len(reload_df) == candidate_rows

assert list(
    reload_df.columns
) == R1_CANDIDATE_COLUMNS

reload_schema = (
    pa.Table.from_pandas(
        reload_df,
        preserve_index=False,
    )
    .schema
)

assert reload_schema.equals(
    candidate_table.schema
), (
    "Serialized candidate schema changed "
    "during round-trip."
)

print(
    "Round-trip rows:",
    f"{len(reload_df):,}",
)

print(
    "Round-trip schema: PASS"
)


# ==============================================================================
# 11. SHA256
# ==============================================================================

def sha256_file_r1(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


candidate_sha256 = (
    sha256_file_r1(
        R1_CANDIDATES_TMP
    )
)

print(
    "Temporary SHA256:",
    candidate_sha256,
)


# ==============================================================================
# 12. RELEASE PYTHON / ARROW REFERENCES BEFORE RENAME
# ==============================================================================

del reload_table
del reload_df
del candidate_table

import gc

gc.collect()


# ==============================================================================
# 13. ATOMIC PUBLICATION
# ==============================================================================

if R1_CANDIDATES_FINAL.exists():

    existing_sha = sha256_file_r1(
        R1_CANDIDATES_FINAL
    )

    if existing_sha == candidate_sha256:

        print(
            "\nExisting final artifact has identical SHA256."
        )

        R1_CANDIDATES_TMP.unlink()

    else:

        raise RuntimeError(
            "Final R1 candidate artifact already exists "
            "with a DIFFERENT SHA256. "
            "Do not overwrite it."
        )

else:

    os.replace(
        R1_CANDIDATES_TMP,
        R1_CANDIDATES_FINAL,
    )


assert R1_CANDIDATES_FINAL.exists(), (
    "Final R1 candidate artifact was not published."
)


# ==============================================================================
# 14. FINAL SHA VERIFICATION
# ==============================================================================

final_sha256 = sha256_file_r1(
    R1_CANDIDATES_FINAL
)

assert final_sha256 == candidate_sha256, (
    "Final candidate SHA256 differs from temporary artifact."
)


# ==============================================================================
# 15. FREEZE MANIFEST
# ==============================================================================

manifest = {
    "artifact": "r1_sparse_candidates",
    "stage": "R1_sparse_retrieval",
    "cell": "6F",
    "status": "FROZEN",

    "rows": int(candidate_rows),
    "responses": int(candidate_responses),
    "sessions": int(candidate_sessions),
    "candidate_turns": int(candidate_turns),

    "top_k": int(R1_TOP_K),

    "columns": list(
        R1_CANDIDATE_COLUMNS
    ),

    "folds": [
        int(x)
        for x in observed_folds
    ],

    "target_isolation": True,

    "duplicate_response_turn_pairs": int(
        duplicate_pairs
    ),

    "sha256": final_sha256,

    "compression": "zstd",

    "python": sys.version,

    "platform": platform.platform(),

    "pandas": pd.__version__,

    "pyarrow": pa.__version__,

    "created_at_local": datetime.now().isoformat(),

    "source_contract": {
        "r0_frozen": True,
        "r0_retrieval_input_ready": True,
        "r1_cell_5_ready": True,
    },

    "rerun_policy": (
        "DO NOT RERUN CELL 6. "
        "Load this frozen artifact instead."
    ),
}


with open(
    R1_FREEZE_MANIFEST,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


assert R1_FREEZE_MANIFEST.exists()


# ==============================================================================
# 16. MANIFEST VALIDATION
# ==============================================================================

with open(
    R1_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    manifest_reload = json.load(
        handle
    )

assert (
    manifest_reload[
        "status"
    ]
    == "FROZEN"
)

assert (
    manifest_reload[
        "sha256"
    ]
    == final_sha256
)

assert (
    manifest_reload[
        "rows"
    ]
    == candidate_rows
)


# ==============================================================================
# 17. FINAL FREEZE STATUS
# ==============================================================================

R1_CELL_6_FROZEN = True
R1_SPARSE_CANDIDATES_FROZEN = True

print("\n" + "=" * 80)
print("R1 CELL 6 FREEZE STATUS")
print("=" * 80)

print(
    "Candidate artifact exists :",
    R1_CANDIDATES_FINAL.exists(),
)

print(
    "Manifest exists            :",
    R1_FREEZE_MANIFEST.exists(),
)

print(
    "Rows                       :",
    f"{candidate_rows:,}",
)

print(
    "SHA256                     :",
    final_sha256,
)

print(
    "Target leakage             :",
    False,
)

print(
    "R1_CELL_6_FROZEN           :",
    R1_CELL_6_FROZEN,
)

print(
    "R1_SPARSE_CANDIDATES_FROZEN:",
    R1_SPARSE_CANDIDATES_FROZEN,
)

print("=" * 80)
print("R1 CELL 6 FREEZE: PASS")
print("=" * 80)

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 6F — FREEZE TOP-K CANDIDATE ARTIFACT

R1 FREEZE PATHS
Freeze root: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen
Candidates: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_sparse_candidates.parquet
Manifest: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_cell6_freeze_manifest.json

CANDIDATE POPULATION
Candidate rows: 1,752,048
Responses covered: 35,072 / 35,072
Candidate sessions: 22,821
Candidate turns: 1,397,075

SERIALIZATION SCHEMA
response_id: string
session_id: string
objective_uid: string
fold: int64
turn_uid: string
role: string
turn_index: int32
word_score: float
char_score: float
math_score: float
sparse_score: float
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1395
Round-trip rows: 1,752,048
Round-trip schema: PASS
Temporary SHA256: 7f408c13f970bd

In [17]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 7 — FROZEN R1 CANDIDATE RELOAD + INTEGRITY VERIFICATION
#
# IMPORTANT:
# This cell MUST NOT recompute retrieval.
# It only reloads and verifies the frozen Cell-6 artifact.
# ==============================================================================

from pathlib import Path
import hashlib
import json
import gc
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 7 — FROZEN R1 CANDIDATE RELOAD + INTEGRITY VERIFICATION")
print("=" * 80)


# ==============================================================================
# 0. RESOLVE PROJECT / R1 PATHS
# ==============================================================================

if "SCRATCH_ROOT" in globals():
    SCRATCH_ROOT_R1 = Path(SCRATCH_ROOT)
else:
    SCRATCH_ROOT_R1 = (
        Path.cwd().parent
        / "scratch_mastery_outputs"
    )

R1_FROZEN_ROOT = (
    SCRATCH_ROOT_R1
    / "02_retrieval"
    / "R1_sparse"
    / "frozen"
)

R1_CANDIDATES_FINAL = (
    R1_FROZEN_ROOT
    / "r1_sparse_candidates.parquet"
)

R1_FREEZE_MANIFEST = (
    R1_FROZEN_ROOT
    / "r1_cell6_freeze_manifest.json"
)


print("\n" + "=" * 80)
print("FROZEN ARTIFACT PATHS")
print("=" * 80)

print(
    "Frozen root:",
    R1_FROZEN_ROOT,
)

print(
    "Candidates:",
    R1_CANDIDATES_FINAL,
)

print(
    "Manifest:",
    R1_FREEZE_MANIFEST,
)


# ==============================================================================
# 1. EXISTENCE CONTRACT
# ==============================================================================

assert R1_FROZEN_ROOT.exists(), (
    "R1 frozen directory does not exist."
)

assert R1_CANDIDATES_FINAL.exists(), (
    "Frozen R1 candidate parquet does not exist."
)

assert R1_FREEZE_MANIFEST.exists(), (
    "R1 Cell-6 freeze manifest does not exist."
)

print("\n" + "=" * 80)
print("ARTIFACT EXISTENCE")
print("=" * 80)

print(
    "Frozen directory :",
    R1_FROZEN_ROOT.exists(),
)

print(
    "Candidate parquet:",
    R1_CANDIDATES_FINAL.exists(),
)

print(
    "Freeze manifest  :",
    R1_FREEZE_MANIFEST.exists(),
)


# ==============================================================================
# 2. MANIFEST LOAD + JSON VALIDATION
# ==============================================================================

with open(
    R1_FREEZE_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    r1_freeze_manifest = json.load(f)


assert isinstance(
    r1_freeze_manifest,
    dict,
)

assert (
    r1_freeze_manifest.get("status")
    == "FROZEN"
), (
    "R1 Cell-6 manifest does not report FROZEN status."
)

assert (
    r1_freeze_manifest.get("artifact")
    == "r1_sparse_candidates"
), (
    "Unexpected frozen artifact identity."
)


print("\n" + "=" * 80)
print("MANIFEST")
print("=" * 80)

print(
    "JSON valid       : True"
)

print(
    "Status           :",
    r1_freeze_manifest["status"],
)

print(
    "Artifact         :",
    r1_freeze_manifest["artifact"],
)

print(
    "Top-K            :",
    r1_freeze_manifest["top_k"],
)


# ==============================================================================
# 3. SHA256 VERIFICATION
# ==============================================================================

def sha256_file_r1_verify(
    path,
    chunk_size=8 * 1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


expected_sha256 = (
    r1_freeze_manifest.get(
        "sha256"
    )
)

assert isinstance(
    expected_sha256,
    str,
), (
    "Manifest does not contain candidate SHA256."
)

actual_sha256 = (
    sha256_file_r1_verify(
        R1_CANDIDATES_FINAL
    )
)

assert actual_sha256 == expected_sha256, (
    "CRITICAL: Frozen candidate artifact SHA256 "
    "does not match the freeze manifest."
)


print("\n" + "=" * 80)
print("SHA256 INTEGRITY")
print("=" * 80)

print(
    "Manifest SHA256:",
    expected_sha256,
)

print(
    "Actual SHA256  :",
    actual_sha256,
)

print(
    "SHA256 match   :",
    actual_sha256 == expected_sha256,
)


# ==============================================================================
# 4. PARQUET METADATA
# ==============================================================================

candidate_parquet_file = (
    pq.ParquetFile(
        R1_CANDIDATES_FINAL
    )
)

candidate_metadata = (
    candidate_parquet_file
    .metadata
)

parquet_rows = (
    candidate_metadata.num_rows
)

parquet_row_groups = (
    candidate_metadata.num_row_groups
)


assert parquet_rows == (
    r1_freeze_manifest["rows"]
), (
    "Parquet row count differs from freeze manifest."
)


print("\n" + "=" * 80)
print("PARQUET METADATA")
print("=" * 80)

print(
    "Rows:",
    f"{parquet_rows:,}",
)

print(
    "Row groups:",
    parquet_row_groups,
)


# ==============================================================================
# 5. LOAD FROZEN CANDIDATES
# ==============================================================================
#
# This is a LOAD operation only.
# No retrieval/scoring is executed.
# ==============================================================================

r1_sparse_candidates_frozen = (
    pd.read_parquet(
        R1_CANDIDATES_FINAL
    )
)


print("\n" + "=" * 80)
print("FROZEN CANDIDATE LOAD")
print("=" * 80)

print(
    "Loaded rows:",
    f"{len(r1_sparse_candidates_frozen):,}",
)


# ==============================================================================
# 6. EXACT SCHEMA CONTRACT
# ==============================================================================

R1_FROZEN_REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]

assert (
    list(
        r1_sparse_candidates_frozen.columns
    )
    ==
    R1_FROZEN_REQUIRED_COLUMNS
), (
    "Frozen candidate schema/column order differs."
)


print("\n" + "=" * 80)
print("SCHEMA CONTRACT")
print("=" * 80)

print(
    "Columns:",
    len(
        r1_sparse_candidates_frozen.columns
    ),
)

print(
    "Schema contract: PASS"
)


# ==============================================================================
# 7. POPULATION CONTRACT
# ==============================================================================

expected_rows = int(
    r1_freeze_manifest[
        "rows"
    ]
)

expected_responses = int(
    r1_freeze_manifest[
        "responses"
    ]
)

expected_sessions = int(
    r1_freeze_manifest[
        "sessions"
    ]
)

expected_top_k = int(
    r1_freeze_manifest[
        "top_k"
    ]
)

assert len(
    r1_sparse_candidates_frozen
) == expected_rows

observed_response_count = (
    r1_sparse_candidates_frozen[
        "response_id"
    ].nunique()
)

observed_session_count = (
    r1_sparse_candidates_frozen[
        "session_id"
    ].nunique()
)

assert (
    observed_response_count
    == expected_responses
)

assert (
    observed_session_count
    == expected_sessions
)


print("\n" + "=" * 80)
print("POPULATION CONTRACT")
print("=" * 80)

print(
    "Rows:",
    f"{len(r1_sparse_candidates_frozen):,}",
    "/",
    f"{expected_rows:,}",
)

print(
    "Responses:",
    f"{observed_response_count:,}",
    "/",
    f"{expected_responses:,}",
)

print(
    "Sessions:",
    f"{observed_session_count:,}",
    "/",
    f"{expected_sessions:,}",
)


# ==============================================================================
# 8. RESPONSE-LEVEL TOP-K CONTRACT
# ==============================================================================

candidate_counts = (
    r1_sparse_candidates_frozen
    .groupby(
        "response_id",
        sort=False,
    )
    .size()
)

assert (
    len(candidate_counts)
    == expected_responses
), (
    "Some responses have no frozen candidates."
)

assert (
    candidate_counts.min()
    > 0
)

assert (
    candidate_counts.max()
    <= expected_top_k
)


print("\n" + "=" * 80)
print("TOP-K CONTRACT")
print("=" * 80)

print(
    "Configured K:",
    expected_top_k,
)

print(
    "Minimum candidates/response:",
    int(candidate_counts.min()),
)

print(
    "Maximum candidates/response:",
    int(candidate_counts.max()),
)

print(
    "Responses with candidates:",
    f"{len(candidate_counts):,}",
)


# ==============================================================================
# 9. RESPONSE / TURN IDENTITY CONTRACT
# ==============================================================================

duplicate_response_turn_pairs = (
    r1_sparse_candidates_frozen[
        [
            "response_id",
            "turn_uid",
        ]
    ]
    .duplicated()
    .sum()
)

assert (
    duplicate_response_turn_pairs == 0
), (
    "Duplicate response_id + turn_uid pairs detected."
)

assert (
    r1_sparse_candidates_frozen[
        "turn_uid"
    ].notna().all()
)

assert (
    r1_sparse_candidates_frozen[
        "response_id"
    ].notna().all()
)

assert (
    r1_sparse_candidates_frozen[
        "session_id"
    ].notna().all()
)


print("\n" + "=" * 80)
print("IDENTITY CONTRACT")
print("=" * 80)

print(
    "Duplicate response/turn pairs:",
    duplicate_response_turn_pairs,
)

print(
    "Identity contract: PASS"
)


# ==============================================================================
# 10. TARGET / LABEL ISOLATION
# ==============================================================================

PROHIBITED_R1_COLUMNS = {
    "target",
    "label",
    "y",
    "correct",
    "is_correct",
    "outcome",
    "score_target",
}

observed_prohibited = (
    PROHIBITED_R1_COLUMNS
    &
    set(
        r1_sparse_candidates_frozen.columns
    )
)

assert observed_prohibited == set(), (
    "Target/evaluation columns found in "
    f"frozen retrieval candidates: "
    f"{sorted(observed_prohibited)}"
)


print("\n" + "=" * 80)
print("TARGET ISOLATION")
print("=" * 80)

print(
    "Prohibited columns found:",
    sorted(observed_prohibited),
)

print(
    "Target leakage: 0"
)


# ==============================================================================
# 11. SCORE VALIDITY
# ==============================================================================

score_columns = [
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
]

for column in score_columns:

    values = (
        r1_sparse_candidates_frozen[
            column
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    assert np.isfinite(
        values
    ).all(), (
        f"Non-finite values detected in {column}."
    )


print("\n" + "=" * 80)
print("SCORE VALIDITY")
print("=" * 80)

for column in score_columns:

    print(
        f"{column:15s}: PASS"
    )


# ==============================================================================
# 12. FOLD CONTRACT
# ==============================================================================

observed_folds = sorted(
    r1_sparse_candidates_frozen[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)

assert observed_folds == [
    0,
    1,
    2,
    3,
    4,
], (
    f"Unexpected folds: {observed_folds}"
)


print("\n" + "=" * 80)
print("FOLD CONTRACT")
print("=" * 80)

print(
    "Observed folds:",
    observed_folds,
)


# ==============================================================================
# 13. ROLE CONTRACT
# ==============================================================================

allowed_roles = {
    "student",
    "tutor",
    "background",
}

observed_roles = set(
    r1_sparse_candidates_frozen[
        "role"
    ]
    .dropna()
    .unique()
)

unexpected_roles = (
    observed_roles
    -
    allowed_roles
)

assert unexpected_roles == set(), (
    f"Unexpected candidate roles: "
    f"{sorted(unexpected_roles)}"
)


# ==============================================================================
# 14. TURN INDEX VALIDITY
# ==============================================================================

assert (
    pd.api.types.is_integer_dtype(
        r1_sparse_candidates_frozen[
            "turn_index"
        ]
    )
)

assert (
    r1_sparse_candidates_frozen[
        "turn_index"
    ]
    >= 0
).all()


# ==============================================================================
# 15. FINAL FREEZE VERIFICATION
# ==============================================================================

R1_CELL_6_FREEZE_VERIFIED = True
R1_SPARSE_CANDIDATES_LOAD_READY = True


print("\n" + "=" * 80)
print("R1 CELL 7 — FROZEN ARTIFACT VERIFICATION")
print("=" * 80)

print(
    "Manifest valid                 : True"
)

print(
    "SHA256 verified                : True"
)

print(
    "Parquet rows verified          : True"
)

print(
    "Schema verified                : True"
)

print(
    "Response population verified   : True"
)

print(
    "Top-K contract verified        : True"
)

print(
    "Identity contract verified     : True"
)

print(
    "Target isolation verified      : True"
)

print(
    "Score validity verified        : True"
)

print(
    "Fold contract verified         : True"
)

print(
    "R1_CELL_6_FREEZE_VERIFIED      :",
    R1_CELL_6_FREEZE_VERIFIED,
)

print(
    "R1_SPARSE_CANDIDATES_LOAD_READY:",
    R1_SPARSE_CANDIDATES_LOAD_READY,
)

assert (
    R1_CELL_6_FREEZE_VERIFIED
    is True
)

assert (
    R1_SPARSE_CANDIDATES_LOAD_READY
    is True
)

print("=" * 80)
print(
    "R1 CELL 7 — FROZEN CANDIDATE VERIFICATION: PASS"
)
print("=" * 80)

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 7 — FROZEN R1 CANDIDATE RELOAD + INTEGRITY VERIFICATION

FROZEN ARTIFACT PATHS
Frozen root: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen
Candidates: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_sparse_candidates.parquet
Manifest: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_cell6_freeze_manifest.json

ARTIFACT EXISTENCE
Frozen directory : True
Candidate parquet: True
Freeze manifest  : True

MANIFEST
JSON valid       : True
Status           : FROZEN
Artifact         : r1_sparse_candidates
Top-K            : 50

SHA256 INTEGRITY
Manifest SHA256: 7f408c13f970bd808247ac5fc9c8d3e34c77aa0cdc4e40855d80e8b76e6d6fee
Actual SHA256  : 7f408c13f970bd808247ac5fc9c8d3e34c77aa0cdc4e40855d80e8b76e6d6fee
SHA256 match   : True

PARQUET METADATA
Rows: 1,752,048
Row groups: 2

FROZEN CANDIDATE LOAD
Loaded rows: 1,752,048

SCH

In [18]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 8 — RETRIEVAL QUALITY DIAGNOSTICS
#
# IMPORTANT:
# This cell NEVER recomputes retrieval.
# It operates only on the frozen R1 candidate artifact.
#
# No target/label is used.
# ==============================================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 8 — RETRIEVAL QUALITY DIAGNOSTICS")
print("=" * 80)


# ==============================================================================
# 0. DEPENDENCY GATES
# ==============================================================================

assert (
    "R1_CELL_6_FREEZE_VERIFIED" in globals()
), (
    "Frozen R1 candidate verification flag is missing."
)

assert (
    R1_CELL_6_FREEZE_VERIFIED is True
), (
    "Frozen R1 candidate artifact was not verified."
)

assert (
    "R1_SPARSE_CANDIDATES_LOAD_READY" in globals()
), (
    "Frozen R1 candidate load flag is missing."
)

assert (
    R1_SPARSE_CANDIDATES_LOAD_READY is True
), (
    "Frozen R1 candidate artifact is not load-ready."
)

assert (
    "r1_sparse_candidates_frozen" in globals()
), (
    "Frozen R1 candidates are not loaded."
)


print("\nR1 frozen candidate dependency : PASS")


# ==============================================================================
# 1. DIAGNOSTIC ROOT
# ==============================================================================

R1_DIAGNOSTIC_ROOT = (
    Path(SCRATCH_ROOT_R1)
    / "02_retrieval"
    / "R1_sparse"
    / "diagnostics"
)

R1_DIAGNOSTIC_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

R1_RESPONSE_DIAGNOSTICS_PATH = (
    R1_DIAGNOSTIC_ROOT
    / "response_diagnostics.parquet"
)

R1_ROLE_DIAGNOSTICS_PATH = (
    R1_DIAGNOSTIC_ROOT
    / "role_diagnostics.parquet"
)

R1_MANUAL_REVIEW_PATH = (
    R1_DIAGNOSTIC_ROOT
    / "manual_review_sample.parquet"
)

R1_DIAGNOSTIC_MANIFEST_PATH = (
    R1_DIAGNOSTIC_ROOT
    / "r1_diagnostic_manifest.json"
)


# ==============================================================================
# 2. BASIC ARTIFACT CONTRACT
# ==============================================================================

candidates = (
    r1_sparse_candidates_frozen
)

required_columns = {
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "word_score",
    "char_score",
    "math_score",
    "sparse_score",
}

assert (
    required_columns
    <= set(candidates.columns)
), (
    "Frozen candidate artifact is missing "
    "diagnostic fields."
)

assert len(candidates) > 0


print("\n" + "=" * 80)
print("INPUT ARTIFACT")
print("=" * 80)

print(
    "Candidate rows:",
    f"{len(candidates):,}",
)

print(
    "Responses:",
    f"{candidates['response_id'].nunique():,}",
)

print(
    "Sessions:",
    f"{candidates['session_id'].nunique():,}",
)

print(
    "Objectives:",
    f"{candidates['objective_uid'].nunique():,}",
)


# ==============================================================================
# 3. RESPONSE-LEVEL CANDIDATE DIAGNOSTICS
# ==============================================================================

response_groups = (
    candidates
    .groupby(
        "response_id",
        sort=False,
    )
)

response_diagnostics = (
    response_groups
    .agg(
        session_id=(
            "session_id",
            "first",
        ),

        objective_uid=(
            "objective_uid",
            "first",
        ),

        fold=(
            "fold",
            "first",
        ),

        candidate_count=(
            "turn_uid",
            "count",
        ),

        unique_candidate_turns=(
            "turn_uid",
            "nunique",
        ),

        max_sparse_score=(
            "sparse_score",
            "max",
        ),

        mean_sparse_score=(
            "sparse_score",
            "mean",
        ),

        median_sparse_score=(
            "sparse_score",
            "median",
        ),

        min_sparse_score=(
            "sparse_score",
            "min",
        ),

        max_word_score=(
            "word_score",
            "max",
        ),

        max_char_score=(
            "char_score",
            "max",
        ),

        max_math_score=(
            "math_score",
            "max",
        ),
    )
    .reset_index()
)


# Candidate-count contract.
assert (
    response_diagnostics[
        "candidate_count"
    ]
    .min()
    > 0
)

assert (
    response_diagnostics[
        "candidate_count"
    ]
    .max()
    <= 50
)

assert (
    response_diagnostics[
        "unique_candidate_turns"
    ]
    ==
    response_diagnostics[
        "candidate_count"
    ]
).all()


# ==============================================================================
# 4. SCORE DISTRIBUTION
# ==============================================================================

score_summary = pd.DataFrame(
    {
        "metric": [
            "sparse_score_mean",
            "sparse_score_median",
            "sparse_score_std",
            "sparse_score_min",
            "sparse_score_max",
            "word_score_mean",
            "char_score_mean",
            "math_score_mean",
        ],

        "value": [
            candidates[
                "sparse_score"
            ].mean(),

            candidates[
                "sparse_score"
            ].median(),

            candidates[
                "sparse_score"
            ].std(),

            candidates[
                "sparse_score"
            ].min(),

            candidates[
                "sparse_score"
            ].max(),

            candidates[
                "word_score"
            ].mean(),

            candidates[
                "char_score"
            ].mean(),

            candidates[
                "math_score"
            ].mean(),
        ],
    }
)


print("\n" + "=" * 80)
print("SPARSE SCORE DISTRIBUTION")
print("=" * 80)

print(
    score_summary.to_string(
        index=False
    )
)


# ==============================================================================
# 5. ROLE COMPOSITION
# ==============================================================================

role_diagnostics = (
    candidates[
        [
            "role",
            "response_id",
            "turn_uid",
            "sparse_score",
        ]
    ]
    .groupby(
        "role",
        sort=True,
    )
    .agg(
        candidate_rows=(
            "turn_uid",
            "count",
        ),

        unique_responses=(
            "response_id",
            "nunique",
        ),

        mean_sparse_score=(
            "sparse_score",
            "mean",
        ),

        median_sparse_score=(
            "sparse_score",
            "median",
        ),

        max_sparse_score=(
            "sparse_score",
            "max",
        ),
    )
    .reset_index()
)

role_diagnostics[
    "candidate_fraction"
] = (
    role_diagnostics[
        "candidate_rows"
    ]
    /
    len(candidates)
)


print("\n" + "=" * 80)
print("ROLE COMPOSITION")
print("=" * 80)

print(
    role_diagnostics.to_string(
        index=False
    )
)


# ==============================================================================
# 6. TOP-K ROLE COMPOSITION
# ==============================================================================

top10 = (
    candidates
    .sort_values(
        [
            "response_id",
            "sparse_score",
            "turn_index",
            "turn_uid",
        ],
        ascending=[
            True,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .groupby(
        "response_id",
        sort=False,
    )
    .head(10)
    .copy()
)

top10_role = (
    top10[
        "role"
    ]
    .value_counts(
        normalize=True
    )
    .rename(
        "fraction"
    )
    .reset_index()
)

top10_role.columns = [
    "role",
    "fraction",
]

print("\n" + "=" * 80)
print("TOP-10 ROLE COMPOSITION")
print("=" * 80)

print(
    top10_role.to_string(
        index=False
    )
)


# ==============================================================================
# 7. OBJECTIVE-LEVEL RETRIEVAL DIAGNOSTICS
# ==============================================================================

objective_diagnostics = (
    candidates
    .groupby(
        "objective_uid",
        sort=False,
    )
    .agg(
        response_count=(
            "response_id",
            "nunique",
        ),

        candidate_rows=(
            "turn_uid",
            "count",
        ),

        mean_sparse_score=(
            "sparse_score",
            "mean",
        ),

        median_sparse_score=(
            "sparse_score",
            "median",
        ),

        max_sparse_score=(
            "sparse_score",
            "max",
        ),
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("OBJECTIVE COVERAGE")
print("=" * 80)

print(
    "Objectives represented:",
    f"{len(objective_diagnostics):,}",
)

print(
    "Expected canonical objectives:",
    "398",
)

assert len(
    objective_diagnostics
) == 398, (
    "Not all canonical objectives are represented "
    "in the frozen R1 candidates."
)


# ==============================================================================
# 8. FOLD-LEVEL DIAGNOSTICS
# ==============================================================================

fold_diagnostics = (
    candidates
    .groupby(
        "fold",
        sort=True,
    )
    .agg(
        candidate_rows=(
            "turn_uid",
            "count",
        ),

        response_count=(
            "response_id",
            "nunique",
        ),

        mean_sparse_score=(
            "sparse_score",
            "mean",
        ),

        median_sparse_score=(
            "sparse_score",
            "median",
        ),

        max_sparse_score=(
            "sparse_score",
            "max",
        ),
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("FOLD DIAGNOSTICS")
print("=" * 80)

print(
    fold_diagnostics.to_string(
        index=False
    )
)

assert (
    fold_diagnostics[
        "fold"
    ].tolist()
    ==
    [0, 1, 2, 3, 4]
)


# ==============================================================================
# 9. TURN-POSITION DIAGNOSTICS
# ==============================================================================

turn_position_summary = (
    candidates
    .groupby(
        "role",
        sort=True,
    )
    .agg(
        mean_turn_index=(
            "turn_index",
            "mean",
        ),

        median_turn_index=(
            "turn_index",
            "median",
        ),

        max_turn_index=(
            "turn_index",
            "max",
        ),

        min_turn_index=(
            "turn_index",
            "min",
        ),
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("TURN POSITION DIAGNOSTICS")
print("=" * 80)

print(
    turn_position_summary.to_string(
        index=False
    )
)


# ==============================================================================
# 10. MANUAL REVIEW SAMPLE
#
# We need actual turn text for qualitative retrieval inspection.
#
# No target is included.
# ==============================================================================

assert (
    "session_turn_index"
    in globals()
), (
    "R0 session_turn_index is not available."
)

session_turn_index_review = (
    session_turn_index[
        [
            "session_id",
            "turn_uid",
            "turn_index",
            "role",
            "text_norm",
            "relative_turn_position",
            "previous_role",
            "next_role",
        ]
    ]
    .copy()
)


# ----------------------------------------------------------------------
# Select deterministic response sample.
#
# We deliberately sample across folds rather than taking only the first
# rows of the dataset.
# ----------------------------------------------------------------------

response_sample = (
    candidates[
        [
            "response_id",
            "fold",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "fold",
        sort=True,
        group_keys=False,
    )
    .apply(
        lambda x:
        x.sample(
            n=min(
                20,
                len(x)
            ),
            random_state=20260814,
        )
    )
    .reset_index(
        drop=True
    )
)


review_base = (
    candidates[
        [
            "response_id",
            "session_id",
            "objective_uid",
            "fold",
            "turn_uid",
            "role",
            "turn_index",
            "word_score",
            "char_score",
            "math_score",
            "sparse_score",
        ]
    ]
    .merge(
        response_sample[
            [
                "response_id"
            ]
        ],
        on="response_id",
        how="inner",
    )
)


# Top 10 candidates per sampled response.
review_base = (
    review_base
    .sort_values(
        [
            "response_id",
            "sparse_score",
            "turn_index",
            "turn_uid",
        ],
        ascending=[
            True,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .groupby(
        "response_id",
        sort=False,
    )
    .head(10)
    .copy()
)


manual_review_sample = (
    review_base
    .merge(
        session_turn_index_review,
        on=[
            "session_id",
            "turn_uid",
            "turn_index",
            "role",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    manual_review_sample[
        "text_norm"
    ].notna().all()
), (
    "Manual review candidate text failed to resolve "
    "from session_turn_index."
)


# ----------------------------------------------------------------------
# Add objective text.
# ----------------------------------------------------------------------

if "objective_catalogue" in globals():

    objective_text_source = (
        objective_catalogue[
            [
                "objective_uid",
                "objective_raw",
            ]
        ]
        .drop_duplicates(
            "objective_uid"
        )
    )

else:

    objective_text_source = (
        pd.read_parquet(
            R0_ROOT
            / "objective_catalogue.parquet"
        )[
            [
                "objective_uid",
                "objective_raw",
            ]
        ]
        .drop_duplicates(
            "objective_uid"
        )
    )


manual_review_sample = (
    manual_review_sample
    .merge(
        objective_text_source,
        on="objective_uid",
        how="left",
        validate="many_to_one",
    )
)


assert (
    manual_review_sample[
        "objective_raw"
    ].notna().all()
)


# Reorder review columns.
manual_review_sample = (
    manual_review_sample[
        [
            "response_id",
            "session_id",
            "fold",
            "objective_uid",
            "objective_raw",
            "turn_uid",
            "turn_index",
            "role",
            "relative_turn_position",
            "previous_role",
            "next_role",
            "word_score",
            "char_score",
            "math_score",
            "sparse_score",
            "text_norm",
        ]
    ]
)


# ==============================================================================
# 11. SAVE DIAGNOSTIC ARTIFACTS
# ==============================================================================

response_diagnostics.to_parquet(
    R1_RESPONSE_DIAGNOSTICS_PATH,
    index=False,
)

role_diagnostics.to_parquet(
    R1_ROLE_DIAGNOSTICS_PATH,
    index=False,
)

manual_review_sample.to_parquet(
    R1_MANUAL_REVIEW_PATH,
    index=False,
)


# ==============================================================================
# 12. DIAGNOSTIC MANIFEST
# ==============================================================================

diagnostic_manifest = {
    "artifact": "r1_sparse_retrieval_diagnostics",
    "stage": "R1_sparse_retrieval",
    "cell": "8",

    "candidate_rows": int(
        len(candidates)
    ),

    "response_count": int(
        candidates[
            "response_id"
        ].nunique()
    ),

    "session_count": int(
        candidates[
            "session_id"
        ].nunique()
    ),

    "objective_count": int(
        candidates[
            "objective_uid"
        ].nunique()
    ),

    "top_k": 50,

    "manual_review_responses": int(
        manual_review_sample[
            "response_id"
        ].nunique()
    ),

    "manual_review_rows": int(
        len(manual_review_sample)
    ),

    "target_used": False,

    "retrieval_recomputed": False,

    "diagnostic_artifacts": {
        "response_diagnostics": str(
            R1_RESPONSE_DIAGNOSTICS_PATH
        ),
        "role_diagnostics": str(
            R1_ROLE_DIAGNOSTICS_PATH
        ),
        "manual_review_sample": str(
            R1_MANUAL_REVIEW_PATH
        ),
    },
}


with open(
    R1_DIAGNOSTIC_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        diagnostic_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ==============================================================================
# 13. FINAL STATUS
# ==============================================================================

R1_CELL_8_READY = True


print("\n" + "=" * 80)
print("R1 CELL 8 STATUS")
print("=" * 80)

print(
    "Frozen candidate rows:",
    f"{len(candidates):,}",
)

print(
    "Responses diagnosed:",
    f"{response_diagnostics.shape[0]:,}",
)

print(
    "Objectives represented:",
    f"{objective_diagnostics.shape[0]:,}",
)

print(
    "Folds:",
    fold_diagnostics[
        "fold"
    ].tolist(),
)

print(
    "Manual review responses:",
    f"{manual_review_sample['response_id'].nunique():,}",
)

print(
    "Manual review rows:",
    f"{len(manual_review_sample):,}",
)

print(
    "Target used:",
    False,
)

print(
    "Retrieval recomputed:",
    False,
)

print(
    "Diagnostic artifacts written:",
    True,
)

print(
    "R1_CELL_8_READY:",
    R1_CELL_8_READY,
)

assert R1_CELL_8_READY is True

print("=" * 80)
print("R1 CELL 8 — RETRIEVAL DIAGNOSTICS: PASS")
print("=" * 80)

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 8 — RETRIEVAL QUALITY DIAGNOSTICS

R1 frozen candidate dependency : PASS

INPUT ARTIFACT
Candidate rows: 1,752,048
Responses: 35,072
Sessions: 22,821
Objectives: 398

SPARSE SCORE DISTRIBUTION
             metric    value
  sparse_score_mean 0.062885
sparse_score_median 0.029547
   sparse_score_std 0.093927
   sparse_score_min 0.000000
   sparse_score_max 1.000000
    word_score_mean 0.046035
    char_score_mean 0.089571
    math_score_mean 0.064980

ROLE COMPOSITION
      role  candidate_rows  unique_responses  mean_sparse_score  median_sparse_score  max_sparse_score  candidate_fraction
background           58222             24683           0.068218             0.030818               1.0            0.033231
   student          529988             35047           0.068945             0.032843               1.0            0.302496
     tutor         1163838             35072           0.059859             0.028279               1.0            0.66

C:\Users\USER\AppData\Local\Temp\ipykernel_10356\1052148336.py:676: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



R1 CELL 8 STATUS
Frozen candidate rows: 1,752,048
Responses diagnosed: 35,072
Objectives represented: 398
Folds: [0, 1, 2, 3, 4]
Manual review responses: 100
Manual review rows: 1,000
Target used: False
Retrieval recomputed: False
Diagnostic artifacts written: True
R1_CELL_8_READY: True
R1 CELL 8 — RETRIEVAL DIAGNOSTICS: PASS


In [19]:
# ==============================================================================
# TRACE THE ACE — R1 SPARSE RETRIEVAL
# CELL 9 — MANUAL REVIEW / TOP-K EVIDENCE INSPECTION
# ==============================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

print("=" * 80)
print("TRACE THE ACE — R1 SPARSE RETRIEVAL")
print("CELL 9 — MANUAL REVIEW / TOP-K EVIDENCE INSPECTION")
print("=" * 80)

# ==============================================================================
# 0. DEPENDENCY CONTRACT
# ==============================================================================

assert R1_CELL_0_READY is True, (
    "R1 Cell 0 must pass."
)

assert R1_CELL_1_READY is True, (
    "R1 Cell 1 must pass."
)

assert R1_CELL_1B_READY is True, (
    "R1 Cell 1B must pass."
)

assert R1_CELL_2_READY is True, (
    "R1 Cell 2 must pass."
)

assert R1_CELL_4_READY is True, (
    "R1 Cell 4 must pass."
)

assert R1_CELL_5_READY is True, (
    "R1 Cell 5 must pass."
)

assert R1_CELL_8_READY is True, (
    "R1 Cell 8 diagnostics must pass."
)

print("\nR1 dependencies: PASS")


# ==============================================================================
# 1. FROZEN ARTIFACT DISCOVERY
# ==============================================================================

R1_FROZEN_ROOT = Path(
    R1_ROOT
) / "frozen"

R1_CANDIDATE_PATH = (
    R1_FROZEN_ROOT
    / "r1_sparse_candidates.parquet"
)

assert R1_CANDIDATE_PATH.exists(), (
    f"Frozen candidate artifact missing:\n{R1_CANDIDATE_PATH}"
)

print("\n" + "=" * 80)
print("FROZEN CANDIDATE")
print("=" * 80)

print("Path:", R1_CANDIDATE_PATH)
print("Exists:", R1_CANDIDATE_PATH.exists())


# ==============================================================================
# 2. DISCOVER DIAGNOSTIC ARTIFACTS
# ==============================================================================

print("\n" + "=" * 80)
print("R1 DIAGNOSTIC ARTIFACT DISCOVERY")
print("=" * 80)

diagnostic_roots = [
    Path(R1_ROOT),
    Path(R1_ROOT) / "diagnostics",
    Path(R1_ROOT) / "quality",
    Path(R1_ROOT) / "artifacts",
]

diagnostic_parquets = []

for root in diagnostic_roots:
    if not root.exists():
        continue

    for p in root.rglob("*.parquet"):
        if p.exists():
            diagnostic_parquets.append(p)

diagnostic_parquets = sorted(
    set(diagnostic_parquets),
    key=lambda p: str(p).lower(),
)

for p in diagnostic_parquets:
    print(" -", p)

print(
    f"\nDiagnostic parquet files found: "
    f"{len(diagnostic_parquets)}"
)


# ==============================================================================
# 3. IDENTIFY MANUAL-REVIEW ARTIFACT
# ==============================================================================

manual_candidates = [
    p
    for p in diagnostic_parquets
    if any(
        token in p.name.lower()
        for token in [
            "manual",
            "review",
            "sample",
            "inspection",
        ]
    )
]

if not manual_candidates:
    # Fallback: inspect parquet schemas and locate likely review artifact.
    print(
        "\nNo filename-based manual-review artifact found."
    )

    for p in diagnostic_parquets:
        try:
            sample = pd.read_parquet(
                p,
                engine="pyarrow",
            )

            cols_lower = {
                str(c).lower()
                for c in sample.columns
            }

            if (
                "response_id" in cols_lower
                and "turn_uid" in cols_lower
                and (
                    "sparse_score" in cols_lower
                    or "word_score" in cols_lower
                )
            ):
                manual_candidates.append(p)

        except Exception as exc:
            print(
                f"Skipped unreadable diagnostic: {p.name} "
                f"({type(exc).__name__})"
            )


assert manual_candidates, (
    "Could not locate the persisted manual-review artifact."
)

# Prefer the smallest sample-like artifact.
manual_candidates = sorted(
    manual_candidates,
    key=lambda p: p.stat().st_size,
)

MANUAL_REVIEW_PATH = manual_candidates[0]

print("\nSelected manual-review artifact:")
print(MANUAL_REVIEW_PATH)


# ==============================================================================
# 4. LOAD MANUAL REVIEW SAMPLE
# ==============================================================================

review_df = pd.read_parquet(
    MANUAL_REVIEW_PATH,
    engine="pyarrow",
)

print("\n" + "=" * 80)
print("MANUAL REVIEW ARTIFACT")
print("=" * 80)

print("Rows    :", f"{len(review_df):,}")
print("Columns :", len(review_df.columns))
print("Fields  :", list(review_df.columns))


# ==============================================================================
# 5. BASIC CONTRACT
# ==============================================================================

required_review_fields = {
    "response_id",
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "sparse_score",
}

missing_review_fields = sorted(
    required_review_fields
    - set(review_df.columns)
)

assert missing_review_fields == [], (
    f"Manual review artifact missing fields: "
    f"{missing_review_fields}"
)

print("\nRequired review fields: PASS")


# ==============================================================================
# 6. RESPONSE / SESSION COVERAGE
# ==============================================================================

response_count = review_df["response_id"].nunique()
session_count = review_df["session_id"].nunique()
turn_count = review_df["turn_uid"].nunique()

print("\n" + "=" * 80)
print("REVIEW COVERAGE")
print("=" * 80)

print("Unique responses :", f"{response_count:,}")
print("Unique sessions  :", f"{session_count:,}")
print("Unique turns     :", f"{turn_count:,}")


# ==============================================================================
# 7. SCORE VALIDITY
# ==============================================================================

score_cols = [
    c
    for c in [
        "word_score",
        "char_score",
        "math_score",
        "sparse_score",
    ]
    if c in review_df.columns
]

print("\n" + "=" * 80)
print("SCORE VALIDITY")
print("=" * 80)

for col in score_cols:
    values = pd.to_numeric(
        review_df[col],
        errors="coerce",
    )

    print(
        f"{col:16s} "
        f"null={values.isna().sum():,} "
        f"min={values.min():.6f} "
        f"max={values.max():.6f}"
    )

    assert values.notna().all(), (
        f"{col} contains null/non-numeric values."
    )


# ==============================================================================
# 8. TOP-10 SAMPLE PER RESPONSE
# ==============================================================================

review_sorted = review_df.sort_values(
    [
        "response_id",
        "sparse_score",
        "turn_index",
    ],
    ascending=[
        True,
        False,
        True,
    ],
)

top10 = (
    review_sorted
    .groupby(
        "response_id",
        sort=False,
    )
    .head(10)
    .copy()
)

print("\n" + "=" * 80)
print("TOP-10 RETRIEVAL SAMPLE")
print("=" * 80)

display_cols = [
    c
    for c in [
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        "turn_uid",
        "role",
        "turn_index",
        "word_score",
        "char_score",
        "math_score",
        "sparse_score",
        "text_norm",
        "content_raw",
    ]
    if c in top10.columns
]

print(
    "Responses represented:",
    f"{top10['response_id'].nunique():,}",
)

print(
    "Rows:",
    f"{len(top10):,}",
)

display(
    top10[display_cols].head(100)
)


# ==============================================================================
# 9. ROLE DISTRIBUTION
# ==============================================================================

print("\n" + "=" * 80)
print("TOP-K ROLE DISTRIBUTION")
print("=" * 80)

role_counts = (
    top10["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="rows")
)

role_counts["share"] = (
    role_counts["rows"]
    / len(top10)
)

display(role_counts)


# ==============================================================================
# 10. SCORE DISTRIBUTION BY RANK
# ==============================================================================

top10_ranked = top10.copy()

top10_ranked["retrieval_rank"] = (
    top10_ranked
    .groupby("response_id")
    .cumcount()
    + 1
)

rank_summary = (
    top10_ranked
    .groupby("retrieval_rank")
    .agg(
        responses=("response_id", "nunique"),
        mean_sparse=("sparse_score", "mean"),
        median_sparse=("sparse_score", "median"),
        min_sparse=("sparse_score", "min"),
        max_sparse=("sparse_score", "max"),
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("RANK-WISE SCORE SUMMARY")
print("=" * 80)

display(rank_summary)


# ==============================================================================
# 11. DUPLICATE / BOILERPLATE CHECK
# ==============================================================================

if "text_norm" in top10.columns:

    text_counts = (
        top10["text_norm"]
        .fillna("")
        .value_counts()
        .head(20)
        .rename_axis("text_norm")
        .reset_index(name="count")
    )

    print("\n" + "=" * 80)
    print("TOP RETRIEVED TEXT REPETITION")
    print("=" * 80)

    display(text_counts)


# ==============================================================================
# 12. CROSS-SESSION CONTAMINATION
# ==============================================================================

candidate_session_map = (
    review_df[
        [
            "response_id",
            "session_id",
        ]
    ]
    .drop_duplicates()
)

# Every response must map to exactly one session.
session_per_response = (
    candidate_session_map
    .groupby("response_id")["session_id"]
    .nunique()
)

cross_session_response_count = int(
    (session_per_response > 1).sum()
)

print("\n" + "=" * 80)
print("SESSION CONTAMINATION")
print("=" * 80)

print(
    "Responses with multiple candidate sessions:",
    cross_session_response_count,
)

assert cross_session_response_count == 0, (
    "Cross-session contamination detected."
)

print("Cross-session contamination: PASS")


# ==============================================================================
# 13. MANUAL REVIEW CONTRACT
# ==============================================================================

assert len(top10) == (
    top10["response_id"].nunique() * 10
), (
    "Top-10 extraction is not exactly 10 rows per response."
)

assert (
    review_df["turn_uid"].nunique()
    == len(review_df)
), (
    "Review artifact contains duplicate turn_uid rows."
)

R1_CELL_9_READY = True

print("\n" + "=" * 80)
print("R1 CELL 9 STATUS")
print("=" * 80)

print(
    "Manual review artifact loaded : True"
)

print(
    "Responses represented         :",
    f"{top10['response_id'].nunique():,}",
)

print(
    "Top-10 rows inspected         :",
    f"{len(top10):,}",
)

print(
    "Cross-session contamination   : PASS"
)

print(
    "R1_CELL_9_READY               :",
    R1_CELL_9_READY,
)

print("=" * 80)

TRACE THE ACE — R1 SPARSE RETRIEVAL
CELL 9 — MANUAL REVIEW / TOP-K EVIDENCE INSPECTION

R1 dependencies: PASS

FROZEN CANDIDATE
Path: d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_sparse_candidates.parquet
Exists: True

R1 DIAGNOSTIC ARTIFACT DISCOVERY
 - d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\char_artifacts\objective_char_index.parquet
 - d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\diagnostics\manual_review_sample.parquet
 - d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\diagnostics\response_diagnostics.parquet
 - d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\diagnostics\role_diagnostics.parquet
 - d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse\frozen\r1_sparse_candidates.parquet
 - d:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retri

,response_id,session_id,objective_uid,fold,turn_uid,role,turn_index,word_score,char_score,math_score,sparse_score,text_norm
0,aaqehuc,ffeeayq,OBJ_ef3c0c8855aafebcfe0a129213ef55f2d6cd72741c...,4,4dbddbbfb6ac41e3ce2ce4d403252e19c9940a75786d59...,student,13,1.000000,1.000000,0.000000,0.800000,Writing tenths as decimals.
1,aaqehuc,ffeeayq,OBJ_ef3c0c8855aafebcfe0a129213ef55f2d6cd72741c...,4,f5552fe14170acd873959f1ba6d4620620054d031cd579...,tutor,109,0.291844,0.271821,0.000000,0.227469,2 tenths.
2,aaqehuc,ffeeayq,OBJ_ef3c0c8855aafebcfe0a129213ef55f2d6cd72741c...,4,a9258e0cd8e6bf7b0704deb5b8caa182765ad738edf5a0...,tutor,113,0.291844,0.269799,0.000000,0.226862,4 tenths.
3,aaqehuc,ffeeayq,OBJ_ef3c0c8855aafebcfe0a129213ef55f2d6cd72741c...,4,2809df149678508f2f449420126673d1ffa00208995a48...,tutor,111,0.291844,0.269105,0.000000,0.226654,3 tenths.
4,aaqehuc,ffeeayq,OBJ_ef3c0c8855aafebcfe0a129213ef55f2d6cd72741c...,4,5eeaeaa22cdcf9e4aa574a8f2b74532be5d2331390fd1e...,background,115,0.291844,0.265619,0.000000,0.225608,5 tenths.
...,...,...,...,...,...,...,...,...,...,...,...,...
95,cawqbzs,dlxhgvf,OBJ_aa3c342a313cd95da1834668088e03ad5dd7117068...,0,59482e8d7864a35d0d1424e2e3706032537ac098f8f947...,tutor,11,0.053559,0.155863,0.000000,0.073538,"Yeah, 2.5 centimeters. So it can be anything. ..."
96,cawqbzs,dlxhgvf,OBJ_aa3c342a313cd95da1834668088e03ad5dd7117068...,0,979cc822585319cc78f4451bbcb0aa5c405a394e2e6e79...,student,145,0.000000,0.016773,0.333333,0.071699,"Let me— wait, let me show you. So I put 2 and ..."
97,cawqbzs,dlxhgvf,OBJ_aa3c342a313cd95da1834668088e03ad5dd7117068...,0,4237f12f26c8bcec836fcfa1ebb5d29ef13ca7b816a134...,student,137,0.000000,0.013345,0.333333,0.070670,"Okay, and then I always have one of those, and..."
98,cawqbzs,dlxhgvf,OBJ_aa3c342a313cd95da1834668088e03ad5dd7117068...,0,eeaf127fb2f5ee27827797b44334c7201cfe0102608677...,tutor,218,0.013604,0.078789,0.166667,0.063772,"Okay, that's fine. So let's proceed. I'll expl..."



TOP-K ROLE DISTRIBUTION


,role,rows,share
0,tutor,636,0.636
1,student,328,0.328
2,background,36,0.036



RANK-WISE SCORE SUMMARY


,retrieval_rank,responses,mean_sparse,median_sparse,min_sparse,max_sparse
0,1,100,0.429646,0.408057,0.020818,0.848961
1,2,100,0.257313,0.244107,0.016196,0.597809
2,3,100,0.203729,0.203549,0.012240,0.535752
3,4,100,0.164186,0.146302,0.006866,0.474542
4,5,100,0.143449,0.118775,0.004240,0.411605
5,6,100,0.128673,0.103241,0.004228,0.355256
6,7,100,0.116689,0.094924,0.003837,0.355169
7,8,100,0.106822,0.084430,0.003628,0.334323
8,9,100,0.100032,0.078056,0.003351,0.327393
9,10,100,0.091968,0.071828,0.003307,0.321908



TOP RETRIEVED TEXT REPETITION


,text_norm,count
0,Mm-hmm.,2
1,rounding.,2
2,The hundredths column.,2
3,Rounding to the nearest tenth.,2
4,"1,000.",2
5,2.,2
6,3 tenths.,1
7,5 tenths.,1
8,We can show tenths using decimals.,1
9,"Yes, so here we will be talking about how we w...",1



SESSION CONTAMINATION
Responses with multiple candidate sessions: 0
Cross-session contamination: PASS

R1 CELL 9 STATUS
Manual review artifact loaded : True
Responses represented         : 100
Top-10 rows inspected         : 1,000
Cross-session contamination   : PASS
R1_CELL_9_READY               : True
